# Spin-up Quick Check Notebook
This notebook is organized for spin-up diagnostics on monthly model output.

## Task 1: Domain map animation of a selected biological variable
- Input file: `dws_500m.3d.201501.nc`
- Goal: animate a selected variable over time on the model map
- Output: an animated GIF (optional) and inline animation preview

## Task 2: Aggregated time series of a selected biological variable for the whole domain
- Cell 2 (Task 2): build and plot the domain-mean trend at a selected layer (4d) or 3d variable
- Input file: `dws_500m.3d.201501.nc`
- Goal: visualize time series of spatially averaged values over the whole domain also for a specific vertical level;
- Output: a time-series line plot

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

from scipy.interpolate import RegularGridInterpolator
from matplotlib.animation import FuncAnimation
from pathlib import Path
from IPython.display import HTML

import pandas as pd

In [ ]:
# meta data:
#! BFM biological model

# pelagic variables (group PelVariables):
#! pelagic  (O)              O2:   Oxygen (mmol/m3)
#! pelagic  (P)              N1:   Phosphate (mmol/m3)
#! pelagic  (N)              N3:   Nitrate (mmol/m3)
#! pelagic  (N)              N4:   Ammonium (mmol/m3)
#! pelagic  (Si)             N5:   Silicate (mmol/m3)
#! pelagic  (R)              N6:   Reduction Equivalents (mmol/m3)
#! pelagic  (N)              O4:   N2-sink (mmol/m3)
#! pelagic  (CNP)            B1:   Pelagic Bacteria
#! pelagic  (CNPSiI)         P1:   Diatoms (group PhytoPlankton))
#! pelagic  (CNPSiI)         P2:   Flagellates (group PhytoPlankton))
#! pelagic  (CNPSiI)         P3:   PicoPhytoPlankton (group PhytoPlankton))
#! pelagic  (CNPSiI)         P4:   Dinoflagellates (group PhytoPlankton))
#! pelagic  (CNP)            Z3:   Carnivorous mesozooplankton (group MesoZooPlankton))
#! pelagic  (CNP)            Z4:   Omnivorous mesozooplankton (group MesoZooPlankton))
#! pelagic  (CNP)            Z5:   Microzooplankton (group MicroZooPlankton))
#! pelagic  (CNP)            Z6:   Heterotrophic nanoflagellates (HNAN) (group MicroZooPlankton))
#! pelagic  (CNPSi)          R1:   Labile Organic Carbon (LOC)
#! pelagic  (C)              R2:   CarboHydrates (sugars)
#! pelagic  (CNPSi)          R6:   Particulate Organic Carbon (POC)
#! pelagic  (C)              R7:   Refractory Disoolved Organic Carbon

# Benthic variables (group BenVariables):
# ! benthic  (CNP)            Y1:   Epibenthos (group BenOrganisms))
# ! benthic  (CNP)            Y2:   Deposit feeders (group BenOrganisms))
# ! benthic  (CNP)            Y3:   Suspension feeders (group BenOrganisms))
# ! benthic  (CNP)            Y4:   Meiobenthos (group ! BenOrganisms))
# ! benthic  (CNP)            Y5:   Benthic predators (group BenOrganisms))
# ! benthic  (CNPSi)          Q1:   Labile organic carbon (group BenDetritus))
# ! benthic  (CNPSi)          Q11:  Labile organic carbon (group BenDetritus))
# ! benthic  (CNPSi)          Q6:   Particulate organic carbon (group BenDetritus))
# ! benthic  (CNP)            H1:   Aerobic benthic bacteria (group BenBacteria))
# ! benthic  (CNP)            H2:   Anaerobic benthic bacteria (group BenBacteria))
# ! benthic  (P)              K1:   Phosphate in oxic layer (group BenthicPhosphate))
# ! benthic  (P)              K11:  Phosphate in denit layer (group BenthicPhosphate))
# ! benthic  (P)              K21:  Phosphate in anoxic layer (group BenthicPhosphate))
# ! benthic  (N)              K4:   Ammonium in oxic layer (group BenthicAmmonium))
# ! benthic  (N)              K14:  Ammonium in denit layer (group BenthicAmmonium))
# ! benthic  (N)              K24:  Ammonium in anoxic layer (group BenthicAmmonium))
# ! benthic  (R)              K6:   Reduction equivalents 
# ! benthic  (M)              D1:   Oxygen penetration depth
# ! benthic  (M)              D2:   Denitrification depth 
# ! benthic  (M)              D6:   Depth distribution factor organic C 
# ! benthic  (M)              D7:   Depth distribution factor organic N
# ! benthic  (M)              D8:   Depth distribution factor organic P
# ! benthic  (M)              D9:   Depth distribution factor organic Si
# ! benthic  (O)              G2:   Benthic O2



# Jetty dataset meta data:
# TSM: mg/L
# C: mg/m3
# TOC: mgC/L
# POC: mgC/L
# DOC: mgC/L

In [ ]:
# setup
# Time series trends for all variables in vars_list over the full year 2015
DATA_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_10')
FILE_PATTERN = 'dws_500m.3d.2015??.nc'
SURFACE_LAYER_INDEX = 10   # Use top 11; surface layer by default
USE_DAILY_MEAN = False     # Set True if you want daily-mean smoothing

Validation_DATA_DIR = Path('/export/lv9/projects/dws/results/validation/pelagic/')
Marsdiep_ts = 'Jetty_ts.csv'
CHLA_VAR = 'Chla'
ELEV_VAR = 'elev'
Bathymetry_VAR = 'bathymetry'  # Preferred name for bathymetry variable; will try alternatives if not found.

Benthic_POC_ts = '20100215_PAM_overview_1974_2009i.xlsx'

# Subdomain index range (Python slice: start inclusive, stop exclusive).
# Salt marsh zone nearby Miedema (unusual discontinuity in derived total Chla)
#X_SLICE = (215, 225)
#Y_SLICE = (133, 136)

# Marsdiep zone
#X_SLICE = (75, 78)
# Y_SLICE = (95, 98)

# Lauwesoog zone
#X_SLICE = (245, 280)
#Y_SLICE = (135, 150)

# Whole subdomain
X_SLICE = (1, 320)
Y_SLICE = (1, 190)

# Chla layer index to inspect and neighboring layers.
CHLA_LAYER_INDEX = 5  # top=11, bottom=1 in your convention

ROLLING_WINDOW = None  # e.g., 3 for smoothing, or None

#PP_csv_path = Validation_DATA_DIR / Marsdiep_PP_ts

# Parameters for model-observation comparison
MODEL_SURFACE_LAYER_INDEX = 10  # top=11, bottom=1 in your convention

# The time series data for Marsdiep is available from the NIOZ Dataverse at http://doi.org/10.25850/nioz/7b.b.5j

In [ ]:
# Helpers
# Reuse opened dataset if available; otherwise open yearly files.

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time dimension found in {da.dims}')

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for d in da.dims:
        if d != time_dim and any(k in d.lower() for k in candidates):
            return d
    return None
   
def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    # Keep first occurrence when duplicate time stamps are present.
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da
    
def _find_bathy_name(ds: xr.Dataset, preferred: str) -> str:
    if preferred in ds.variables:
        return preferred
    candidates = ('bathymetry', 'depth', 'h', 'H', 'bathy', 'bat', 'topo', 'd', 'water_depth')
    for name in candidates:
        if name in ds.variables:
            return name
    raise KeyError(
        f"Bathymetry variable not found. Tried '{preferred}' and {candidates}. "
        f"Available vars include: {list(ds.variables)[:30]}"
    )

def _to_float(values) -> np.ndarray:
    if np.ma.isMaskedArray(values):
        values = np.ma.filled(values, np.nan)
    return np.asarray(values, dtype=float)

def _pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables:
            return name
    return None

def _maybe_smooth(series: xr.DataArray, time_dim: str) -> xr.DataArray:
    out = series
    if USE_DAILY_MEAN:
        out = out.resample({time_dim: '1D'}).mean(skipna=True)
    if ROLLING_WINDOW is not None:
        if ROLLING_WINDOW < 1:
            raise ValueError('ROLLING_WINDOW must be >= 1 or None')
        out = out.rolling({time_dim: ROLLING_WINDOW}, center=True).mean()
    return out

In [ ]:
# List of variables for analysis and visualization
vars_list = [
    'elev',
    #'temp',
    #'salt',
    #'O2o', 
    'netPPm2',
    #'N1p',
    #'N3n',
    #'N4n',
    #'N5s',
    #'N6r',
#         'B1c',
#         'Bac',
          'P1c',
#     'P2c',
#     'P3c',
#         'P4c',
#         'P5c',
#         'P6c',
#	      'P1l',
#         'P2l',
#         'P3l',
#         'P4l',
#         'P5l',
#         'P6l',
#         'Z2c',
#         'Z3c',
#         'Z4c',
#         'Z5c',
#         'Z6c',
#         'R1c',
#         'R2c',
#         'R3c',
#          'R6c',
#         'RZc',
#         'Q1c',
#         'Q11c',
#          'Q6c',
          'Chla',
#          'H1c',
#          'H2c',
#          'HNc',
#          'Hac',
          'Y1c',
          'Y2c',
          'Y3c',
#          'Y4c',
          'Y5c',
#          'Yy3c',
#          'K6r',
#          'K16r',
#          'K26r',
#          'K5s',
#          'K15s',
#          'K3n',
#          'K4n',
#         'K13n',
#          'K14n',
#          'K24n',
#          'K1p',
#          'K11p',
#         'K21p',
#          'D1m',
#          'D2m',
#          'O3c',
#          'pCO2',
#          'CO2',
#          'HCO3',
#          'CO3',
#          'pH',
#          'Ac'
#          'G3h'
#          'G13h'
#          'G23h'
#          'G3c'
#          'G13c'
#          'G23c'
#          'G14n'
#          'Acae'
#          'Acan'
#          'DICae'
#          'DICan'
#          'pHae'
#          'pHan'
#          'pCO2ae'
#          'pCO2an'
#          'G3h'
#          'G13h'
          'BP1c',
#          'ETW',
          'ESS',
#          'irrenh',
#          'turenh',
'xEPS'      
]

In [ ]:

# Compare EMOaaS model output with RWS field measurements
# Parameter SPM — depth-averaged model ESS vs observed SPM at Vliestroom and Marsdiep Noord

from pyproj import Transformer
from scipy.spatial import cKDTree

VLIESM_ts = 'VLIESM_cleanA056.csv'
MARSDND_ts = 'MARSDND_cleanA056.csv'

# RD New (EPSG:28992) → WGS84 lon/lat
_rd2wgs = Transformer.from_crs("EPSG:28992", "EPSG:4326", always_xy=True)

# Station RD coordinates (m) and derived lon/lat
STATIONS = {
    'Vliestroom':     {'rd_x': 139850, 'rd_y': 591900},
    'Marsdiep Noord': {'rd_x': 112200, 'rd_y': 555250},
}
for _info in STATIONS.values():
    _info['lon'], _info['lat'] = _rd2wgs.transform(_info['rd_x'], _info['rd_y'])

SPM_FILES = {
    'Vliestroom':     Validation_DATA_DIR / 'Field' / VLIESM_ts,
    'Marsdiep Noord': Validation_DATA_DIR / 'Field' / MARSDND_ts,
}

# Load observations; datum format is DD-MM-YY (e.g. 08-01-86)
obs_dfs = {}
for _station, _fpath in SPM_FILES.items():
    if not _fpath.exists():
        print(f"Warning: {_fpath} not found — skipping {_station}")
        continue
    _df = pd.read_csv(_fpath, na_values=['NA', ''])
    _df['timestamp'] = pd.to_datetime(_df['datum'], format='%d-%m-%y', errors='coerce')
    _df = _df.dropna(subset=['timestamp', 'waarde'])
    _df['SPM_mg_m3'] = _df['waarde'] * 1000  # mg/L → mg/m³
    obs_dfs[_station] = _df.sort_values('timestamp')
    print(f"{_station}: {len(_df)} observations, "
          f"{_df['timestamp'].dt.year.min()}–{_df['timestamp'].dt.year.max()}")

for _name, _info in STATIONS.items():
    print(f"{_name}: lon={_info['lon']:.4f}°, lat={_info['lat']:.4f}°")

# Model dataset — use spinup_datasets if already loaded, else open from DATA_DIR
try:
    _ds_spm = list(spinup_datasets.values())[-1]
except NameError:
    _spm_files = sorted(DATA_DIR.glob(FILE_PATTERN))
    if not _spm_files:
        raise FileNotFoundError(f"No files found: {FILE_PATTERN} in {DATA_DIR}")
    _ds_spm = xr.open_mfdataset(
        _spm_files,
        combine='nested', concat_dim='time', decode_times=True,
        data_vars='minimal', coords='minimal', compat='override', join='override',
    )

if 'ESS' not in _ds_spm.variables:
    raise KeyError(f"'ESS' not found in model. Available: {sorted(_ds_spm.data_vars)}")

_ess = _ds_spm['ESS'].squeeze(drop=True)
_td  = _find_time_dim(_ess)
_zd  = _find_vertical_dim(_ess, _td)
_ess = _drop_duplicate_time(_ess, _td)

# Depth-average over all layers
if _zd is not None:
    _ess = _ess.mean(dim=_zd, skipna=True)

# Model grid lon/lat for nearest-neighbour lookup
_lon_name = _pick_coord_name(_ds_spm, ('lonc', 'lon', 'longitude'))
_lat_name = _pick_coord_name(_ds_spm, ('latc', 'lat', 'latitude'))
if _lon_name is None or _lat_name is None:
    raise ValueError("Model does not have recognisable lon/lat coordinates.")

_lon2d    = _ds_spm[_lon_name].values
_lat2d    = _ds_spm[_lat_name].values
_lon_flat = _lon2d.ravel()
_lat_flat = _lat2d.ravel()
_vmask    = np.isfinite(_lon_flat) & np.isfinite(_lat_flat)
_vidx     = np.where(_vmask)[0]
_tree     = cKDTree(np.column_stack([_lon_flat[_vmask], _lat_flat[_vmask]]))

_h_dims = [d for d in _ess.dims if d != _td]
if len(_h_dims) != 2:
    raise ValueError(f"Expected 2 horizontal dims in ESS, got: {_ess.dims}")
_y_dim, _x_dim = _h_dims

# Plot
fig, axes = plt.subplots(len(STATIONS), 1, figsize=(13, 4 * len(STATIONS)),
                         constrained_layout=True)
if len(STATIONS) == 1:
    axes = [axes]

for ax, (_station, _info) in zip(axes, STATIONS.items()):
    _, _nn = _tree.query([[_info['lon'], _info['lat']]])
    _fi  = _vidx[_nn[0]]
    _iy, _ix = np.unravel_index(_fi, _lon2d.shape)

    _ts = _ess.isel({_y_dim: int(_iy), _x_dim: int(_ix)})
    _ts = _ts.where(_ts >= 0)

    ax.plot(_ts[_td].values, _ts.values, lw=1.6, color='tab:blue',
            label='Model ESS (depth avg)')

    if _station in obs_dfs:
        _obs = obs_dfs[_station][obs_dfs[_station]['timestamp'].dt.year == 2015]
        ax.scatter(_obs['timestamp'], _obs['SPM_mg_m3'],
                   s=18, color='tab:orange', alpha=0.85, label='Obs SPM', zorder=3)

    ax.set_xlim(pd.Timestamp('2015-01-01'), pd.Timestamp('2016-01-01'))
    _units = _ds_spm['ESS'].attrs.get('units', 'mg/m³')
    ax.set_ylabel(f'SPM [{_units}]')
    ax.set_title(f"{_station}  (lon={_info['lon']:.3f}°, lat={_info['lat']:.3f}°)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper right')
    ax.set_xlabel('Time')

fig.suptitle('EMOaaS model ESS vs RWS SPM measurements', fontsize=13, fontweight='bold')
fig.autofmt_xdate()
plt.show()


In [ ]:

# Compare EMOaaS model output with TrilaWatt model output for 2015 hydrodynamics
# curl -O https://dl.datenrepository.baw.de/7000/B3955.02.04.70237/Hydrodynamik/2015/transport_2015_nl.nc
# curl -O https://dl.datenrepository.baw.de/7000/B3955.02.04.70237/Hydrodynamik/2015/tide_2015_nl.nc

TrilaWatt_data_dir = Path('/export/lv9/projects/dws/results/validation/pelagic/TrilaWatt/Hydrodynamik/2015')
TrilaWatt_transport = TrilaWatt_data_dir / 'transport_2015_nl.nc'  # salt, temp, ESS
TrilaWatt_tide      = TrilaWatt_data_dir / 'tides_2015_nl.nc'       # sea surface height

# EMOaaS Model output
DATA_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_10/')
FILE_PATTERN = 'dws_500m.3d.2015??.nc'
SURFACE_LAYER_INDEX = 10   # Use top 11; surface layer by default
USE_DAILY_MEAN = False     # Set True if you want daily-mean smoothing

# Lat/lon bounding box to subset TrilaWatt to the Wadden Sea domain
TW_LON_RANGE = (4.5, 9.0)
TW_LAT_RANGE = (52.5, 55.5)

# TrilaWatt variables to compare and their counterparts in the model
HYDRO_VAR_MAP = [
    ('salt', 'sea_water_salinity_2d',             'Salinity [psu]'),
    ('temp', 'sea_water_temperature_2d',           'Temperature [°C]'),
    ('ESS',  'suspended_sediment_concentration_2d','Suspended Sediment [mg/m³]'),
    #('elev', 'sea_surface_height_2d',                 'Sea Surface Height [m]'),
]

# Multiply TrilaWatt values by these factors to match model units before comparison
TW_UNIT_SCALE = {
    'suspended_sediment_concentration_2d': 1e6,  # kg/m³ → mg/m³
}

def _tw_decode_and_subset(ds):
    """Decode CF hours-since time and subset to Wadden Sea lon/lat box."""
    t0  = np.datetime64('2015-01-01T00:00:00', 'ns')
    dts = (ds['time'].values * 3600 * 1e9).astype('timedelta64[ns]')
    ds  = ds.assign_coords(time=t0 + dts)
    lm  = (ds['lon'] >= TW_LON_RANGE[0]) & (ds['lon'] <= TW_LON_RANGE[1])
    lt  = (ds['lat'] >= TW_LAT_RANGE[0]) & (ds['lat'] <= TW_LAT_RANGE[1])
    return ds.sel(lon=lm, lat=lt)

# --- Load TrilaWatt transport (salt, temp, ESS) ---
ds_tw = _tw_decode_and_subset(
    xr.open_dataset(TrilaWatt_transport, decode_times=False, mask_and_scale=True)
)

# --- Load SSH from tide file and merge ---
ds_tide = _tw_decode_and_subset(
    xr.open_dataset(TrilaWatt_tide, decode_times=False, mask_and_scale=True)
)
if 'sea_surface_height_2d' in ds_tide.variables:
    ds_tw = xr.merge([ds_tw, ds_tide[['sea_surface_height_2d']]], join='inner')
else:
    print('Warning: sea_surface_height_2d not found in tide file')

print(f'TrilaWatt time  : {str(ds_tw.time.values[0])[:10]} → {str(ds_tw.time.values[-1])[:10]}  ({ds_tw.dims["time"]} steps, ~20 min)')
print(f'TrilaWatt subset: lon [{float(ds_tw.lon.min()):.2f}, {float(ds_tw.lon.max()):.2f}], '
      f'lat [{float(ds_tw.lat.min()):.2f}, {float(ds_tw.lat.max()):.2f}]')
print(f'TrilaWatt variables: {list(ds_tw.data_vars)}')

# Load model 2015 data and plot comparison with TrilaWatt

model_files = sorted(DATA_DIR.glob(FILE_PATTERN))
if not model_files:
    raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {DATA_DIR}')

ds_2015 = xr.open_mfdataset(
    model_files,
    combine='nested',
    concat_dim='time',
    decode_times=True,
    data_vars='minimal',
    coords='minimal',
    compat='override',
    join='override',
)

In [ ]:


# Resample TrilaWatt from 20-min to daily means for salinity, water temperature, suspended sediment, and sea surface height, but not for the water levels
ds_tw_daily = ds_tw.resample(time='1D').mean(skipna=True)

fig, axes = plt.subplots(
    len(HYDRO_VAR_MAP), 1,
    figsize=(14, 4 * len(HYDRO_VAR_MAP)),
    sharex=True,
    constrained_layout=True,
)
if len(HYDRO_VAR_MAP) == 1:
    axes = [axes]

for ax, (model_var, tw_var, ylabel) in zip(axes, HYDRO_VAR_MAP):

    # --- TrilaWatt: spatial mean, scaled to model units ---
    scale = TW_UNIT_SCALE.get(tw_var, 1.0)
    tw_series = ds_tw_daily[tw_var].mean(dim=['lat', 'lon'], skipna=True) * scale

    # --- Model: surface layer, full-domain spatial mean ---
    if model_var not in ds_2015.variables:
        ax.set_title(f'{model_var} — not found in model output')
        continue

    da = ds_2015[model_var].squeeze(drop=True)
    time_dim = _find_time_dim(da)
    z_dim = _find_vertical_dim(da, time_dim)

    if z_dim is not None:
        da = da.isel({z_dim: MODEL_SURFACE_LAYER_INDEX})

    da = _drop_duplicate_time(da, time_dim)
    spatial_dims = [d for d in da.dims if d != time_dim]
    model_series = da.mean(dim=spatial_dims, skipna=True)

    ax.plot(
        model_series[time_dim].values, model_series.values,
        lw=1.4, color='tab:blue', label='EMOaaS (daily)',
    )
    ax.plot(
        tw_series['time'].values, tw_series.values,
        lw=1.2, color='tab:orange', alpha=0.85,
        label='TrilaWatt (daily mean of 20-min data)',
    )

    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Date')
fig.suptitle('EMOaaS vs TrilaWatt — Hydrodynamic validation 2015', fontsize=13, fontweight='bold')
plt.show()

# save the resampled daily TrilaWatt data to a NetCDF file for future use
output_nc_path = TrilaWatt_data_dir / 'trilawatt_daily_2015.nc'
ds_tw_daily.to_netcdf(output_nc_path)
print(f"Saved daily-mean TrilaWatt data to {output_nc_path}")

In [ ]:
# Animated daily bias: EMOaaS − TrilaWatt (bias panel only)

from scipy.interpolate import griddata
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Target grid: TrilaWatt regular lat/lon
tw_lon   = ds_tw.lon.values
tw_lat   = ds_tw.lat.values
tw_lon2d, tw_lat2d = np.meshgrid(tw_lon, tw_lat)

lon_name = _pick_coord_name(ds_2015, ('lonc', 'lon', 'longitude'))
lat_name = _pick_coord_name(ds_2015, ('latc', 'lat', 'latitude'))
if lon_name is None or lat_name is None:
    raise ValueError('Model does not have recognisable lon/lat coordinates.')

model_lons = ds_2015[lon_name].values.ravel()
model_lats = ds_2015[lat_name].values.ravel()

# Resample model to daily means
ds_2015_daily = ds_2015.resample(time='1D').mean(skipna=True)

# Find overlapping dates
tw_dates     = pd.DatetimeIndex(ds_tw_daily.time.values).normalize()
model_dates  = pd.DatetimeIndex(ds_2015_daily.time.values).normalize()
common_dates = tw_dates.intersection(model_dates)
print(f'Common daily steps: {len(common_dates)}  ({common_dates[0].date()} → {common_dates[-1].date()})')

# ── Pre-compute bias stacks for all variables ──────────────────────────────
bias_stacks = {}
blim_global = {}

for model_var, tw_var, label in HYDRO_VAR_MAP:
    if model_var not in ds_2015.variables or tw_var not in ds_tw_daily.variables:
        bias_stacks[(model_var, tw_var)] = None
        continue

    scale = TW_UNIT_SCALE.get(tw_var, 1.0)

    da_d = ds_2015_daily[model_var].squeeze(drop=True)
    td   = _find_time_dim(da_d)
    zd   = _find_vertical_dim(da_d, td)
    if zd is not None:
        da_d = da_d.isel({zd: MODEL_SURFACE_LAYER_INDEX})

    frames = []
    for i, date in enumerate(common_dates):
        model_snap = da_d.sel({td: date}, method='nearest').values.ravel()
        tw_snap    = ds_tw_daily[tw_var].sel(time=date, method='nearest').values * scale

        valid  = np.isfinite(model_lons) & np.isfinite(model_lats) & np.isfinite(model_snap)
        interp = griddata(
            (model_lons[valid], model_lats[valid]),
            model_snap[valid],
            (tw_lon2d, tw_lat2d),
            method='linear',
        )
        frames.append(interp - tw_snap)
        if (i + 1) % 60 == 0:
            print(f'  {model_var}: {i + 1}/{len(common_dates)} days done')

    stack = np.stack(frames, axis=0)
    bias_stacks[(model_var, tw_var)] = stack
    blim_global[(model_var, tw_var)] = float(np.nanpercentile(np.abs(stack), 95))
    print(f'{model_var} done — symmetric colour limit: ±{blim_global[(model_var, tw_var)]:.4f}')

# ── Build figure: one bias panel per variable ──────────────────────────────
n_vars = len(HYDRO_VAR_MAP)
fig, axes = plt.subplots(1, n_vars, figsize=(7 * n_vars, 5), constrained_layout=True)
if n_vars == 1:
    axes = [axes]

meshes     = []
title_objs = []

for ax, (model_var, tw_var, label) in zip(axes, HYDRO_VAR_MAP):
    unit  = label.split('[')[-1].rstrip(']') if '[' in label else ''
    stack = bias_stacks.get((model_var, tw_var))
    blim  = blim_global.get((model_var, tw_var), 1.0) or 1.0

    if stack is None:
        ax.text(0.5, 0.5, f'{model_var}\nnot found', ha='center', va='center', transform=ax.transAxes)
        meshes.append(None)
        title_objs.append(None)
        continue

    im = ax.pcolormesh(tw_lon2d, tw_lat2d, stack[0], vmin=-blim, vmax=blim, cmap='RdBu_r', shading='auto')
    fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02, label=unit)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    t = ax.set_title(f'Bias EMOaaS − TrilaWatt\n{label}', fontsize=10)
    meshes.append(im)
    title_objs.append(t)

sup = fig.suptitle(f'Daily bias EMOaaS vs TrilaWatt — {common_dates[0].date()}',
                   fontsize=13, fontweight='bold')


def _update(frame):
    date = common_dates[frame]
    for im, (model_var, tw_var, _) in zip(meshes, HYDRO_VAR_MAP):
        if im is None:
            continue
        im.set_array(bias_stacks[(model_var, tw_var)][frame])
    sup.set_text(f'Daily bias EMOaaS vs TrilaWatt — {date.date()}')
    return [m for m in meshes if m is not None] + [sup]


anim = FuncAnimation(fig, _update, frames=len(common_dates), interval=200, blit=False)
HTML(anim.to_jshtml())


In [ ]:
# Animated annual averaged bias: EMOaaS − TrilaWatt (bias panel only)
# based on the previous daily bias stacks, compute the annual mean bias for each variable and plot it.

years        = pd.DatetimeIndex(common_dates).year
unique_years = sorted(set(years))
print(f'Years available: {unique_years}')

# Pre-compute annual mean bias stacks from daily stacks
annual_stacks = {}
annual_lims   = {}

for (model_var, tw_var), stack in bias_stacks.items():
    if stack is None:
        annual_stacks[(model_var, tw_var)] = None
        annual_lims[(model_var, tw_var)]   = 1.0
        continue
    yr_means = [np.nanmean(stack[years == yr], axis=0) for yr in unique_years]
    ann = np.stack(yr_means, axis=0)   # shape: (n_years, n_lat, n_lon)
    annual_stacks[(model_var, tw_var)] = ann
    annual_lims[(model_var, tw_var)]   = float(np.nanpercentile(np.abs(ann), 95)) or 1.0

# ── Build figure: one bias panel per variable ──────────────────────────────
fig2, axes2 = plt.subplots(1, n_vars, figsize=(7 * n_vars, 5), constrained_layout=True)
if n_vars == 1:
    axes2 = [axes2]

meshes2 = []

for ax, (model_var, tw_var, label) in zip(axes2, HYDRO_VAR_MAP):
    unit  = label.split('[')[-1].rstrip(']') if '[' in label else ''
    stack = annual_stacks.get((model_var, tw_var))
    blim  = annual_lims.get((model_var, tw_var), 1.0) or 1.0

    if stack is None:
        ax.text(0.5, 0.5, f'{model_var}\nnot found', ha='center', va='center', transform=ax.transAxes)
        meshes2.append(None)
        continue

    im = ax.pcolormesh(tw_lon2d, tw_lat2d, stack[0], vmin=-blim, vmax=blim, cmap='RdBu_r', shading='auto')
    fig2.colorbar(im, ax=ax, fraction=0.035, pad=0.02, label=unit)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'Annual mean bias EMOaaS − TrilaWatt\n{label}', fontsize=10)
    meshes2.append(im)

sup2 = fig2.suptitle(f'Annual mean bias EMOaaS vs TrilaWatt — {unique_years[0]}',
                     fontsize=13, fontweight='bold')


def _update_annual(frame):
    yr = unique_years[frame]
    for im, (model_var, tw_var, _) in zip(meshes2, HYDRO_VAR_MAP):
        if im is None:
            continue
        im.set_array(annual_stacks[(model_var, tw_var)][frame])
    sup2.set_text(f'Annual mean bias EMOaaS vs TrilaWatt — {yr}')
    return [m for m in meshes2 if m is not None] + [sup2]


anim2 = FuncAnimation(fig2, _update_annual, frames=len(unique_years), interval=600, blit=False)
HTML(anim2.to_jshtml())


In [ ]:
if 'ESS' not in _ds_spm.variables:
    raise KeyError(f"'ESS' not found in model. Available: {sorted(_ds_spm.data_vars)}")

_ess = _ds_spm['ESS'].squeeze(drop=True)
_td  = _find_time_dim(_ess)
_zd  = _find_vertical_dim(_ess, _td)
_ess = _drop_duplicate_time(_ess, _td)

# Depth-average over all layers
if _zd is not None:
    _ess = _ess.mean(dim=_zd, skipna=True)

# Model grid lon/lat for nearest-neighbour lookup
_lon_name = _pick_coord_name(_ds_spm, ('lonc', 'lon', 'longitude'))
_lat_name = _pick_coord_name(_ds_spm, ('latc', 'lat', 'latitude'))
if _lon_name is None or _lat_name is None:
    raise ValueError("Model does not have recognisable lon/lat coordinates.")

_lon2d    = _ds_spm[_lon_name].values
_lat2d    = _ds_spm[_lat_name].values
_lon_flat = _lon2d.ravel()
_lat_flat = _lat2d.ravel()
_vmask    = np.isfinite(_lon_flat) & np.isfinite(_lat_flat)
_vidx     = np.where(_vmask)[0]
_tree     = cKDTree(np.column_stack([_lon_flat[_vmask], _lat_flat[_vmask]]))

_h_dims = [d for d in _ess.dims if d != _td]
if len(_h_dims) != 2:
    raise ValueError(f"Expected 2 horizontal dims in ESS, got: {_ess.dims}")
_y_dim, _x_dim = _h_dims



# read the resampled daily TrilaWatt data from a NetCDF file
output_nc_path = TrilaWatt_data_dir / 'trilawatt_daily_2015.nc'
ds_tw_daily = xr.open_dataset(output_nc_path)
print(f"Loaded daily-mean TrilaWatt data from {output_nc_path}")

_tw_ssc_scale = TW_UNIT_SCALE.get('suspended_sediment_concentration_2d', 1.0)
_tw_ssc_daily = ds_tw_daily['suspended_sediment_concentration_2d']

# ── One subplot per site ──────────────────────────────────────────────────
_sites   = [s for s in sorted(mwtl_2015['locatie.code'].unique()) if s in _site_xy.index]
_n_sites = len(_sites)
fig, axes = plt.subplots(_n_sites, 1, figsize=(13, 4 * _n_sites), constrained_layout=True)
axes = np.atleast_1d(axes)

for ax, _site in zip(axes, _sites):
    _slon = _site_xy.loc[_site, 'lon']
    _slat = _site_xy.loc[_site, 'lat']

    # EMOaaS nearest grid point, resample to daily
    _, _nn = _tree_mwtl.query([[_slon, _slat]])
    _fi = _vi_mwtl[_nn[0]]
    _iy, _ix = np.unravel_index(_fi, _lon2d_mwtl.shape)
    _emo_ts    = _ess_da.isel({_hy_mwtl: int(_iy), _hx_mwtl: int(_ix)})
    _emo_daily = _emo_ts.resample({_td_mwtl: '1D'}).mean(skipna=True)
    _emo_times = pd.to_datetime(_emo_daily[_td_mwtl].values)
    _emo_vals  = _emo_daily.values
    _mask_2015 = pd.DatetimeIndex(_emo_times).year == 2015
    ax.plot(_emo_times[_mask_2015], _emo_vals[_mask_2015],
            lw=1.2, color='tab:blue', alpha=0.85, label='EMOaaS ESS (depth avg, daily)')

    # TrilaWatt nearest 5x5 box, daily
    if _tw_ssc_daily is not None:
        _ti = int(np.argmin(np.abs(ds_tw.lon.values - _slon)))
        _tj = int(np.argmin(np.abs(ds_tw.lat.values - _slat)))
        _tw_pt = _tw_ssc_daily.isel(
            lon=slice(max(0, _ti - 2), _ti + 3),
            lat=slice(max(0, _tj - 2), _tj + 3),
        ).mean(dim=['lon', 'lat'], skipna=True) * _tw_ssc_scale
        _tw_df2  = _tw_pt.to_dataframe(name='ssc').dropna()
        _tw_2015 = _tw_df2['ssc'][_tw_df2.index.year == 2015]
        ax.plot(_tw_2015.index, _tw_2015.values,
                lw=0.9, color='tab:green', alpha=0.8, label='TrilaWatt SSC (daily, mg/m³)')

    # MWTL observations for this site
    _obs_site = mwtl_2015[mwtl_2015['locatie.code'] == _site]
    ax.scatter(_obs_site['date'], _obs_site['numeriekewaarde'],
               s=20, color='tab:orange', alpha=0.85, linewidths=0,
               label='MWTL obs', zorder=3)

    _units = ds_2015['ESS'].attrs.get('units', 'mg/m³')
    ax.set_xlim(pd.Timestamp('2015-01-01'), pd.Timestamp('2016-01-01'))
    ax.set_ylabel(f'SPM [{_units}]')
    ax.set_title(f"{_site}  (lon={_slon:.3f}°, lat={_slat:.3f}°)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper right')
    ax.set_xlabel('Time (2015)')

fig.suptitle('SPM: EMOaaS vs TrilaWatt vs MWTL observations (2015)',
             fontsize=13, fontweight='bold')
fig.autofmt_xdate()
plt.show()

In [ ]:
# Compare EMOaaS & TrilaWatt model outputs with MWTL field measurements
# Suspended particulate matter validation at multiple sites for 2015

from scipy.spatial import cKDTree
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

MWTL_Turbidity_2015 = 'MWTL_Turbidity.csv'
_fpath = Validation_DATA_DIR / 'Field' / MWTL_Turbidity_2015

# ------------------------------------------------------------------
# 1. Load & prepare MWTL observations
# ------------------------------------------------------------------
with open(_fpath, 'r') as f:
    header = f.readline().strip().split(',')
    print("Column names in MWTL_Turbidity.csv:")
    for col in header:
        print(f"  - {col}")

mwtl_raw = pd.read_csv(_fpath)

# tijdstip format: 1986-01-07T12:45:00Z — parse UTC then strip timezone
mwtl_raw['tijdstip'] = pd.to_datetime(mwtl_raw['tijdstip'], utc=True).dt.tz_convert(None)
mwtl_raw['date'] = mwtl_raw['tijdstip'].dt.date

# Daily mean of numeriekewaarde per site
mwtl_daily = (
    mwtl_raw
    .groupby(['locatie.code', 'date'])['numeriekewaarde']
    .mean()
    .reset_index()
)
mwtl_daily['date'] = pd.to_datetime(mwtl_daily['date'])

# Convert mg/L → mg/m³
mwtl_daily['SPM_mg_m3'] = mwtl_daily['numeriekewaarde'] * 1000

# ------------------------------------------------------------------
# 2. Parse geom → lon/lat (one unique location per site)
# ------------------------------------------------------------------
def parse_geom(geom_str):
    """Parse 'POINT (lon lat)' → (lon, lat)"""
    if isinstance(geom_str, str) and geom_str.startswith('POINT'):
        # Remove the "POINT " prefix and the surrounding parentheses
        coords_str = geom_str[6:].strip().strip('()')
        coords = coords_str.split()
        return float(coords[0]), float(coords[1])
    raise ValueError(f"Unexpected geom format: {geom_str}")

# One unique location per site
site_geom = (
    mwtl_raw[['locatie.code', 'geom']]
    .drop_duplicates('locatie.code')
    .set_index('locatie.code')
)

_site_xy = pd.DataFrame(
    [parse_geom(g) for g in site_geom['geom']],
    index=site_geom.index,
    columns=['lon', 'lat']
)

mwtl_daily = mwtl_daily.join(_site_xy, on='locatie.code')
mwtl_2015 = mwtl_daily[mwtl_daily['date'].dt.year == 2015].copy()

print(f"\nSites: {sorted(mwtl_daily['locatie.code'].unique())}")
print(f"Daily records in 2015: {len(mwtl_2015)}")
print("\nSite coordinates:")
print(_site_xy)

# ------------------------------------------------------------------
# 3. Prepare EMOaaS model (depth-averaged ESS)
# ------------------------------------------------------------------
# Assumes ds_2015 is already loaded (equivalent to the working script’s _ds_spm)
if 'ESS' not in ds_2015.variables:
    raise KeyError(f"'ESS' not found in model. Available: {sorted(ds_2015.data_vars)}")

_ess = ds_2015['ESS'].squeeze(drop=True)
_td  = _find_time_dim(_ess)
_zd  = _find_vertical_dim(_ess, _td)
_ess = _drop_duplicate_time(_ess, _td)

# Depth-average over all layers
if _zd is not None:
    _ess = _ess.mean(dim=_zd, skipna=True)

_ess = _ess.where(_ess >= 0)          # mask negative values

# Horizontal dimension names
_h_dims = [d for d in _ess.dims if d != _td]
if len(_h_dims) != 2:
    raise ValueError(f"Expected 2 horizontal dims in ESS, got: {_ess.dims}")
_y_dim, _x_dim = _h_dims

# ------------------------------------------------------------------
# 4. Build KDTree on EMOaaS model grid (same pattern as working script)
# ------------------------------------------------------------------
_lon_name = _pick_coord_name(ds_2015, ('lonc', 'lon', 'longitude'))
_lat_name = _pick_coord_name(ds_2015, ('latc', 'lat', 'latitude'))
if _lon_name is None or _lat_name is None:
    raise ValueError("Model does not have recognisable lon/lat coordinates.")

_lon2d    = ds_2015[_lon_name].values
_lat2d    = ds_2015[_lat_name].values
_lon_flat = _lon2d.ravel()
_lat_flat = _lat2d.ravel()
_vmask    = np.isfinite(_lon_flat) & np.isfinite(_lat_flat)
_vidx     = np.where(_vmask)[0]
_tree     = cKDTree(np.column_stack([_lon_flat[_vmask], _lat_flat[_vmask]]))

# ------------------------------------------------------------------
# 5. Plot – one subplot per site
# ------------------------------------------------------------------
_sites   = [s for s in sorted(mwtl_2015['locatie.code'].unique()) if s in _site_xy.index]
_n_sites = len(_sites)

fig, axes = plt.subplots(_n_sites, 1, figsize=(13, 4 * _n_sites),
                         constrained_layout=True)
axes = np.atleast_1d(axes)

for ax, _site in zip(axes, _sites):
    _slon = _site_xy.loc[_site, 'lon']
    _slat = _site_xy.loc[_site, 'lat']

    # ---- EMOaaS nearest grid point, daily mean ----
    _, _nn = _tree.query([[_slon, _slat]])
    _fi = _vidx[_nn[0]]
    _iy, _ix = np.unravel_index(_fi, _lon2d.shape)

    _emo_ts = _ess.isel({_y_dim: int(_iy), _x_dim: int(_ix)})
    _emo_daily = _emo_ts.resample({_td: '1D'}).mean(skipna=True)

    _emo_times = pd.to_datetime(_emo_daily[_td].values)
    _emo_vals  = _emo_daily.values
    _mask_2015 = pd.DatetimeIndex(_emo_times).year == 2015

    ax.plot(_emo_times[_mask_2015], _emo_vals[_mask_2015],
            lw=1.4, color='tab:blue', alpha=0.85,
            label='EMOaaS ESS (depth-avg, daily)')

    # ---- TrilaWatt nearest 5×5 box (if available) ----
    if '_tw_ssc_daily' in globals() and _tw_ssc_daily is not None:
        _ti = int(np.argmin(np.abs(ds_tw.lon.values - _slon)))
        _tj = int(np.argmin(np.abs(ds_tw.lat.values - _slat)))
        _tw_pt = (
            _tw_ssc_daily
            .isel(
                lon=slice(max(0, _ti - 2), _ti + 3),
                lat=slice(max(0, _tj - 2), _tj + 3),
            )
            .mean(dim=['lon', 'lat'], skipna=True)
            * _tw_ssc_scale
        )
        _tw_df = _tw_pt.to_dataframe(name='ssc').dropna()
        _tw_2015 = _tw_df['ssc'][_tw_df.index.year == 2015]
        ax.plot(_tw_2015.index, _tw_2015.values,
                lw=1.0, color='tab:green', alpha=0.8,
                label='TrilaWatt SSC (daily)')

    # ---- MWTL observations ----
    _obs_site = mwtl_2015[mwtl_2015['locatie.code'] == _site]
    ax.scatter(_obs_site['date'], _obs_site['SPM_mg_m3'],
               s=22, color='tab:orange', alpha=0.85, linewidths=0,
               label='MWTL obs', zorder=3)

    # Styling
    _units = ds_2015['ESS'].attrs.get('units', 'mg/m³')
    ax.set_xlim(pd.Timestamp('2015-01-01'), pd.Timestamp('2016-01-01'))
    ax.set_ylabel(f'SPM [{_units}]')
    ax.set_title(f"{_site}  (lon={_slon:.3f}°, lat={_slat:.3f}°)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper right')
    ax.set_xlabel('Time (2015)')

fig.suptitle('SPM: EMOaaS vs TrilaWatt vs MWTL observations (2015)',
             fontsize=13, fontweight='bold')
fig.autofmt_xdate()
plt.show()

In [ ]:
# Animated daily bias: EMOaaS ESS − TrilaWatt SSC  (single panel + monitoring sites)
# Robust version that diagnoses empty interpolations

from scipy.interpolate import griddata
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ------------------------------------------------------------------
# Load pre-saved daily TrilaWatt data
# ------------------------------------------------------------------
output_nc_path = TrilaWatt_data_dir / 'trilawatt_daily_2015.nc'
ds_tw_daily = xr.open_dataset(output_nc_path)
print(f"Loaded daily TrilaWatt data from {output_nc_path}")
print(f"  variables : {list(ds_tw_daily.data_vars)}")

TW_SSC_VAR = 'suspended_sediment_concentration_2d'

# ------------------------------------------------------------------
# Target grid
# ------------------------------------------------------------------
tw_lon = ds_tw_daily.lon.values
tw_lat = ds_tw_daily.lat.values
tw_lon2d, tw_lat2d = np.meshgrid(tw_lon, tw_lat)

# ------------------------------------------------------------------
# EMOaaS coordinates  (make sure they match the data shape)
# ------------------------------------------------------------------
lon_name = _pick_coord_name(ds_2015, ('lonc', 'lon', 'longitude'))
lat_name = _pick_coord_name(ds_2015, ('latc', 'lat', 'latitude'))
if lon_name is None or lat_name is None:
    raise ValueError('Model does not have recognisable lon/lat coordinates.')

model_lons = ds_2015[lon_name].values.ravel()
model_lats = ds_2015[lat_name].values.ravel()
print(f"Model lon/lat finite points: {np.isfinite(model_lons).sum()} / {model_lons.size}")

# ------------------------------------------------------------------
# Daily means of EMOaaS
# ------------------------------------------------------------------
ds_2015_daily = ds_2015.resample(time='1D').mean(skipna=True)

tw_dates     = pd.DatetimeIndex(ds_tw_daily.time.values).normalize()
model_dates  = pd.DatetimeIndex(ds_2015_daily.time.values).normalize()
common_dates = tw_dates.intersection(model_dates)
print(f'Common daily steps: {len(common_dates)}  '
      f'({common_dates[0].date()} → {common_dates[-1].date()})')

# ------------------------------------------------------------------
# Prepare ESS (depth-averaged)
# ------------------------------------------------------------------
if 'ESS' not in ds_2015_daily.variables:
    raise KeyError(f"'ESS' not found. Available: {sorted(ds_2015_daily.data_vars)}")

ess_da = ds_2015_daily['ESS'].squeeze(drop=True)
td = _find_time_dim(ess_da)
zd = _find_vertical_dim(ess_da, td)
if zd is not None:
    ess_da = ess_da.mean(dim=zd, skipna=True)
ess_da = ess_da.where(ess_da >= 0)

print(f"ESS dims after processing: {ess_da.dims}")
print(f"ESS shape: {ess_da.shape}")

# Unit scale
tw_ssc_scale = TW_UNIT_SCALE.get(TW_SSC_VAR, 1000.0)

# ------------------------------------------------------------------
# Pre-compute bias stack with diagnostics
# ------------------------------------------------------------------
frames = []
valid_dates = []          # only keep days that produced a valid interpolation
n_empty = 0

for i, date in enumerate(common_dates):
    model_snap = ess_da.sel({td: date}, method='nearest').values.ravel()

    # Safety: shapes must match
    if model_snap.size != model_lons.size:
        raise ValueError(
            f"Shape mismatch on {date.date()}: "
            f"data {model_snap.size} vs lon/lat {model_lons.size}"
        )

    valid = (np.isfinite(model_lons) &
             np.isfinite(model_lats) &
             np.isfinite(model_snap))

    n_valid = valid.sum()
    if n_valid < 3:                       # need at least a few points for linear
        n_empty += 1
        if n_empty <= 5:                  # print only the first few warnings
            print(f"  WARNING: only {n_valid} valid points on {date.date()} – skipping")
        continue

    tw_snap = (ds_tw_daily[TW_SSC_VAR]
               .sel(time=date, method='nearest')
               .values * tw_ssc_scale)

    interp = griddata(
        (model_lons[valid], model_lats[valid]),
        model_snap[valid],
        (tw_lon2d, tw_lat2d),
        method='linear',
    )
    frames.append(interp - tw_snap)
    valid_dates.append(date)

    if (i + 1) % 60 == 0:
        print(f'  ESS: {i + 1}/{len(common_dates)} days processed '
              f'({len(frames)} kept, {n_empty} empty)')

if len(frames) == 0:
    raise RuntimeError(
        "No valid interpolation frames were produced. "
        "Check that ESS contains finite values and that lon/lat "
        "coordinates match the horizontal dimensions of ESS."
    )

bias_stack = np.stack(frames, axis=0)
common_dates = pd.DatetimeIndex(valid_dates)   # replace with the kept dates
blim = float(np.nanpercentile(np.abs(bias_stack), 95))
print(f'\nKept {len(common_dates)} days  |  colour limit ±{blim:.1f}')
print(f'Skipped {n_empty} empty days')

# ------------------------------------------------------------------
# Monitoring sites
# ------------------------------------------------------------------
try:
    site_lons = _site_xy['lon'].values
    site_lats = _site_xy['lat'].values
except NameError:
    site_lons = site_lats = []

# ------------------------------------------------------------------
# Figure
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)

im = ax.pcolormesh(
    tw_lon2d, tw_lat2d, bias_stack[0],
    vmin=-blim, vmax=blim, cmap='RdBu_r', shading='auto'
)
fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02, label='mg/m³')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

ax.scatter(site_lons, site_lats,
           s=40, c='k', marker='o', edgecolors='white', linewidths=0.8,
           zorder=5, label='MWTL sites')
ax.legend(loc='upper right', fontsize=9)

title = ax.set_title(
    f'Bias EMOaaS ESS − TrilaWatt SSC\n{common_dates[0].date()}',
    fontsize=12
)
fig.suptitle('Daily bias EMOaaS vs TrilaWatt (ESS only)', fontsize=13, fontweight='bold')


def _update(frame):
    date = common_dates[frame]
    im.set_array(bias_stack[frame].ravel())
    title.set_text(f'Bias EMOaaS ESS − TrilaWatt SSC\n{date.date()}')
    return [im, title]


anim = FuncAnimation(fig, _update, frames=len(common_dates), interval=200, blit=False)
HTML(anim.to_jshtml())

In [ ]:
# Scatter: MWTL obs (x) vs EMOaaS & TrilaWatt (y) — SPM / ESS, one panel per site

from scipy.spatial import cKDTree

# ------------------------------------------------------------------
# Make sure we have the daily TrilaWatt data and the site coordinates
# ------------------------------------------------------------------
if 'ds_tw_daily' not in globals():
    ds_tw_daily = xr.open_dataset(TrilaWatt_data_dir / 'trilawatt_daily_2015.nc')

TW_SSC_VAR = 'suspended_sediment_concentration_2d'
tw_ssc_scale = TW_UNIT_SCALE.get(TW_SSC_VAR, 1000.0)   # adjust if needed

# ------------------------------------------------------------------
# EMOaaS ESS (depth-averaged, daily)
# ------------------------------------------------------------------
ds_2015_daily = ds_2015.resample(time='1D').mean(skipna=True)

ess_da = ds_2015_daily['ESS'].squeeze(drop=True)
td = _find_time_dim(ess_da)
zd = _find_vertical_dim(ess_da, td)
if zd is not None:
    ess_da = ess_da.mean(dim=zd, skipna=True)
ess_da = ess_da.where(ess_da >= 0)

lon_name = _pick_coord_name(ds_2015, ('lonc', 'lon', 'longitude'))
lat_name = _pick_coord_name(ds_2015, ('latc', 'lat', 'latitude'))
model_lons = ds_2015[lon_name].values
model_lats = ds_2015[lat_name].values
_lon_flat = model_lons.ravel()
_lat_flat = model_lats.ravel()
_vmask = np.isfinite(_lon_flat) & np.isfinite(_lat_flat)
_vidx = np.where(_vmask)[0]
_tree = cKDTree(np.column_stack([_lon_flat[_vmask], _lat_flat[_vmask]]))

_h_dims = [d for d in ess_da.dims if d != td]
_y_dim, _x_dim = _h_dims

# ------------------------------------------------------------------
# Sites that have 2015 observations
# ------------------------------------------------------------------
_sites = [s for s in sorted(mwtl_2015['locatie.code'].unique()) if s in _site_xy.index]
_n = len(_sites)

fig, axes = plt.subplots(1, _n, figsize=(5.2 * _n, 5), constrained_layout=True)
axes = np.atleast_1d(axes)

for ax, _site in zip(axes, _sites):
    _slon = _site_xy.loc[_site, 'lon']
    _slat = _site_xy.loc[_site, 'lat']

    # ---- observations for this site (already daily) ----
    _obs = mwtl_2015[mwtl_2015['locatie.code'] == _site].copy()
    if _obs.empty:
        ax.set_title(f"{_site}\nno 2015 obs")
        continue

    _obs_times = pd.DatetimeIndex(_obs['date'])
    _obs_vals  = _obs['SPM_mg_m3'].values

    # ---- EMOaaS nearest grid point ----
    _, _nn = _tree.query([[_slon, _slat]])
    _fi = _vidx[_nn[0]]
    _iy, _ix = np.unravel_index(_fi, model_lons.shape)

    _emo_ts = ess_da.isel({_y_dim: int(_iy), _x_dim: int(_ix)})
    _emo_times = pd.DatetimeIndex(pd.to_datetime(_emo_ts[td].values))
    _emo_vals  = _emo_ts.values

    # nearest-neighbour in time
    _t_idx = np.array([int(np.argmin(np.abs(_emo_times - t))) for t in _obs_times])
    _emo_matched = _emo_vals[_t_idx]

    ax.scatter(_obs_vals, _emo_matched,
               s=22, color='tab:blue', alpha=0.75, linewidths=0,
               label='EMOaaS', zorder=3)

    _all = np.concatenate([_obs_vals, _emo_matched])

    # ---- TrilaWatt nearest 5×5 box ----
    if TW_SSC_VAR in ds_tw_daily.variables:
        _ti = int(np.argmin(np.abs(ds_tw_daily.lon.values - _slon)))
        _tj = int(np.argmin(np.abs(ds_tw_daily.lat.values - _slat)))
        _tw_pt = (
            ds_tw_daily[TW_SSC_VAR]
            .isel(
                lon=slice(max(0, _ti - 2), _ti + 3),
                lat=slice(max(0, _tj - 2), _tj + 3),
            )
            .mean(dim=['lon', 'lat'], skipna=True)
            * tw_ssc_scale
        )
        _tw_df = _tw_pt.to_dataframe(name='ssc').dropna()
        _tw_2015 = _tw_df['ssc'][_tw_df.index.year == 2015]
        _tw_times = pd.DatetimeIndex(_tw_2015.index)

        _tw_idx = np.array([int(np.argmin(np.abs(_tw_times - t))) for t in _obs_times])
        _tw_matched = _tw_2015.values[_tw_idx]

        ax.scatter(_obs_vals, _tw_matched,
                   s=22, color='tab:green', alpha=0.75, linewidths=0,
                   label='TrilaWatt', zorder=2)
        _all = np.concatenate([_all, _tw_matched])

    # ---- 1:1 line & limits ----
    _finite = _all[np.isfinite(_all)]
    if len(_finite) == 0:
        ax.set_title(f"{_site}\nno finite data")
        continue

    _pad = 0.05 * (_finite.max() - _finite.min() + 1)
    _lim = [_finite.min() - _pad, _finite.max() + _pad]
    ax.plot(_lim, _lim, 'k--', lw=1.0, label='1:1')
    ax.set_xlim(_lim)
    ax.set_ylim(_lim)
    ax.set_aspect('equal')

    ax.set_xlabel('MWTL obs [mg/m³]')
    ax.set_ylabel('Model [mg/m³]')
    ax.set_title(f"{_site}\n(lon={_slon:.2f}, lat={_slat:.2f})")
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper left', fontsize=8)

fig.suptitle('SPM scatter: model vs MWTL observations (2015, daily)',
             fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# Scatter: all MWTL sites combined — obs (x) vs EMOaaS & TrilaWatt (y)

fig, ax = plt.subplots(figsize=(7, 6.5), constrained_layout=True)

all_obs, all_emo, all_tw = [], [], []

for _site in _sites:
    _slon = _site_xy.loc[_site, 'lon']
    _slat = _site_xy.loc[_site, 'lat']

    _obs = mwtl_2015[mwtl_2015['locatie.code'] == _site]
    if _obs.empty:
        continue

    _obs_times = pd.DatetimeIndex(_obs['date'])
    _obs_vals  = _obs['SPM_mg_m3'].values

    # ---- EMOaaS ----
    _, _nn = _tree.query([[_slon, _slat]])
    _fi = _vidx[_nn[0]]
    _iy, _ix = np.unravel_index(_fi, model_lons.shape)

    _emo_ts    = ess_da.isel({_y_dim: int(_iy), _x_dim: int(_ix)})
    _emo_times = pd.DatetimeIndex(pd.to_datetime(_emo_ts[td].values))
    _emo_vals  = _emo_ts.values

    _t_idx       = np.array([int(np.argmin(np.abs(_emo_times - t))) for t in _obs_times])
    _emo_matched = _emo_vals[_t_idx]

    all_obs.append(_obs_vals)
    all_emo.append(_emo_matched)

    # ---- TrilaWatt ----
    if TW_SSC_VAR in ds_tw_daily.variables:
        _ti = int(np.argmin(np.abs(ds_tw_daily.lon.values - _slon)))
        _tj = int(np.argmin(np.abs(ds_tw_daily.lat.values - _slat)))
        _tw_pt = (
            ds_tw_daily[TW_SSC_VAR]
            .isel(
                lon=slice(max(0, _ti - 2), _ti + 3),
                lat=slice(max(0, _tj - 2), _tj + 3),
            )
            .mean(dim=['lon', 'lat'], skipna=True)
            * tw_ssc_scale
        )
        _tw_df   = _tw_pt.to_dataframe(name='ssc').dropna()
        _tw_2015 = _tw_df['ssc'][_tw_df.index.year == 2015]
        _tw_times = pd.DatetimeIndex(_tw_2015.index)

        _tw_idx     = np.array([int(np.argmin(np.abs(_tw_times - t))) for t in _obs_times])
        _tw_matched = _tw_2015.values[_tw_idx]
        all_tw.append(_tw_matched)

# Concatenate
all_obs = np.concatenate(all_obs)
all_emo = np.concatenate(all_emo)
all_tw  = np.concatenate(all_tw) if all_tw else np.array([])

# Plot
ax.scatter(all_obs, all_emo,
           s=18, color='tab:blue', alpha=0.65, linewidths=0,
           label=f'EMOaaS (n={np.isfinite(all_emo).sum()})', zorder=3)

if len(all_tw):
    ax.scatter(all_obs, all_tw,
               s=18, color='tab:green', alpha=0.65, linewidths=0,
               label=f'TrilaWatt (n={np.isfinite(all_tw).sum()})', zorder=2)

# 1:1 line
_all = np.concatenate([all_obs, all_emo, all_tw]) if len(all_tw) else np.concatenate([all_obs, all_emo])
_finite = _all[np.isfinite(_all)]
_pad = 0.05 * (_finite.max() - _finite.min() + 1)
_lim = [_finite.min() - _pad, _finite.max() + _pad]

ax.plot(_lim, _lim, 'k--', lw=1.2, label='1:1')
ax.set_xlim(_lim)
ax.set_ylim(_lim)
ax.set_aspect('equal')

ax.set_xlabel('MWTL obs [mg/m³]')
ax.set_ylabel('Model [mg/m³]')
ax.set_title('SPM: all MWTL sites combined (2015, daily)')
ax.grid(True, alpha=0.25)
ax.legend(loc='upper left', fontsize=9)

fig.suptitle('SPM scatter: model vs MWTL observations — all sites', 
             fontsize=13, fontweight='bold')
plt.show()

# ------------------------------------------------------------------
# Compute R² (only on finite pairs)
# ------------------------------------------------------------------

def _r2(obs, mod):
    mask = np.isfinite(obs) & np.isfinite(mod)
    if mask.sum() < 2:
        return np.nan
    o, m = obs[mask], mod[mask]
    ss_res = np.sum((o - m) ** 2)
    ss_tot = np.sum((o - np.mean(o)) ** 2)
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan


r2_emo = _r2(all_obs, all_emo)
r2_tw  = _r2(all_obs, all_tw) if len(all_tw) else np.nan
print(f"EMOaaS   R² = {r2_emo:.4f}")
print(f"TrilaWatt R² = {r2_tw:.4f}")

In [ ]:
# ------------------------------------------------------------------
# Compute R² (only on finite pairs)
# ------------------------------------------------------------------

def _r2(obs, mod):
    mask = np.isfinite(obs) & np.isfinite(mod)
    if mask.sum() < 2:
        return np.nan
    o, m = obs[mask], mod[mask]
    ss_res = np.sum((o - m) ** 2)
    ss_tot = np.sum((o - np.mean(o)) ** 2)
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan


r2_emo = _r2(all_obs, all_emo)
r2_tw  = _r2(all_obs, all_tw) if len(all_tw) else np.nan

# ------------------------------------------------------------------
# Plot
# ------------------------------------------------------------------
ax.scatter(all_obs, all_emo,
           s=18, color='tab:blue', alpha=0.65, linewidths=0,
           label=f'EMOaaS  (R² = {r2_emo:.3f}, n={np.isfinite(all_emo).sum()})',
           zorder=3)

if len(all_tw):
    ax.scatter(all_obs, all_tw,
               s=18, color='tab:green', alpha=0.65, linewidths=0,
               label=f'TrilaWatt (R² = {r2_tw:.3f}, n={np.isfinite(all_tw).sum()})',
               zorder=2)

# 1:1 line
_all = np.concatenate([all_obs, all_emo, all_tw]) if len(all_tw) else np.concatenate([all_obs, all_emo])
_finite = _all[np.isfinite(_all)]
_pad = 0.05 * (_finite.max() - _finite.min() + 1)
_lim = [_finite.min() - _pad, _finite.max() + _pad]

ax.plot(_lim, _lim, 'k--', lw=1.2, label='1:1')
ax.set_xlim(_lim)
ax.set_ylim(_lim)
ax.set_aspect('equal')

ax.set_xlabel('MWTL obs [mg/m³]')
ax.set_ylabel('Model [mg/m³]')
ax.set_title('SPM: all MWTL sites combined (2015, daily)')
ax.grid(True, alpha=0.25)
ax.legend(loc='upper left', fontsize=9)

fig.suptitle('SPM scatter: model vs MWTL observations — all sites',
             fontsize=13, fontweight='bold')
plt.show()

print(f"EMOaaS   R² = {r2_emo:.4f}")
print(f"TrilaWatt R² = {r2_tw:.4f}")

In [ ]:
# Compare EMOaaS & TrilaWatt model outputs with RWS field measurements
# water level validation at Marsdiep and Schiermonnikoog for 2015
# All sources filtered to 00:00:00 (midnight) to match EMOaaS output frequency.

wl_marsdiep_2015        = 'wl_marsdiep_2015.csv'
wl_schiermonnikoog_2015 = 'wl_schiermonnikoog_2015.csv'

from scipy.spatial import cKDTree

STATIONS_WL = {
    'Marsdiep':        {'lon': 4.785, 'lat': 52.964, 'file': wl_marsdiep_2015},
    'Schiermonnikoog': {'lon': 6.202, 'lat': 53.469, 'file': wl_schiermonnikoog_2015},
}

# Load RWS observations, keep only midnight records
wl_obs_dfs = {}
for _stn, _meta in STATIONS_WL.items():
    _fpath = Validation_DATA_DIR / 'Field' / _meta['file']
    if not _fpath.exists():
        print(f"Warning: {_fpath} not found — skipping {_stn}")
        continue
    _df = pd.read_csv(_fpath, na_values=['NA', ''], sep=';')
    _df['timestamp'] = pd.to_datetime(
        _df['WAARNEMINGDATUM'].astype(str) + ' ' + _df['WAARNEMINGTIJD'].astype(str),
        dayfirst=True, errors='coerce'
    )
    _df = _df.dropna(subset=['timestamp', 'NUMERIEKEWAARDE'])
    _df['wl_m'] = _df['NUMERIEKEWAARDE'] / 100  # cm NAP → m
    _df = _df[(_df['timestamp'].dt.hour == 0) & (_df['timestamp'].dt.minute == 0) & (_df['timestamp'].dt.second == 0)]
    wl_obs_dfs[_stn] = _df.sort_values('timestamp')
    print(f"{_stn}: {len(_df)} midnight observations, "
          f"{_df['timestamp'].dt.year.min()}–{_df['timestamp'].dt.year.max()}")

# Build KDTree on EMOaaS model grid
_ln_wl    = _pick_coord_name(ds_2015, ('lonc', 'lon', 'longitude'))
_la_wl    = _pick_coord_name(ds_2015, ('latc', 'lat', 'latitude'))
_lon2d_wl = ds_2015[_ln_wl].values
_lat2d_wl = ds_2015[_la_wl].values
_lf_wl    = _lon2d_wl.ravel()
_ltf_wl   = _lat2d_wl.ravel()
_vm_wl    = np.isfinite(_lf_wl) & np.isfinite(_ltf_wl)
_vi_wl    = np.where(_vm_wl)[0]
_tree_wl  = cKDTree(np.column_stack([_lf_wl[_vm_wl], _ltf_wl[_vm_wl]]))

# Prepare EMOaaS elev DataArray (already at 00:00:00 only)
if 'elev' not in ds_2015.variables:
    raise KeyError(f"'elev' not in model; available: {sorted(ds_2015.data_vars)}")
_elev_da = ds_2015['elev'].squeeze(drop=True)
_td_wl   = _find_time_dim(_elev_da)
_zd_wl   = _find_vertical_dim(_elev_da, _td_wl)
if _zd_wl is not None:
    _elev_da = _elev_da.isel({_zd_wl: MODEL_SURFACE_LAYER_INDEX})
_elev_da    = _drop_duplicate_time(_elev_da, _td_wl)
_hy_wl, _hx_wl = [d for d in _elev_da.dims if d != _td_wl]

# One panel per station
fig, axes = plt.subplots(len(STATIONS_WL), 1,
                         figsize=(13, 4 * len(STATIONS_WL)),
                         constrained_layout=True)
axes = np.atleast_1d(axes)

for _idx, (_stn, _meta) in enumerate(STATIONS_WL.items()):
    ax = axes[_idx]

    # EMOaaS: point extraction at native 00:00:00 frequency
    _, _nn = _tree_wl.query([[_meta['lon'], _meta['lat']]])
    _fi = _vi_wl[_nn[0]]
    _iy, _ix = np.unravel_index(_fi, _lon2d_wl.shape)
    _model_ts    = _elev_da.isel({_hy_wl: int(_iy), _hx_wl: int(_ix)})
    _model_times = pd.to_datetime(_model_ts[_td_wl].values)
    _model_vals  = _model_ts.values

    ax.plot(_model_times, _model_vals,
            marker='o', ms=3, lw=1.0, color='tab:blue', alpha=0.85, label='EMOaaS (00:00)')

    # TrilaWatt: 5x5 box average, filtered to 00:00:00 only
    if 'sea_surface_height_2d' in ds_tw.variables:
        _ti = int(np.argmin(np.abs(ds_tw.lon.values - _meta['lon'])))
        _tj = int(np.argmin(np.abs(ds_tw.lat.values - _meta['lat'])))
        _tw_box = ds_tw['sea_surface_height_2d'].isel(
            lon=slice(max(0, _ti - 2), _ti + 3),
            lat=slice(max(0, _tj - 2), _tj + 3),
        ).mean(dim=['lon', 'lat'], skipna=True)
        _tw_df = _tw_box.to_dataframe(name='ssh').dropna()
        _tw_midnight = _tw_df['ssh'][
            (_tw_df.index.year == 2015) &
            (_tw_df.index.hour == 0) & (_tw_df.index.minute == 0) & (_tw_df.index.second == 0)
        ]
        ax.plot(_tw_midnight.index, _tw_midnight.values,
                marker='s', ms=3, lw=0.9, color='tab:green', alpha=0.8, label='TrilaWatt (00:00)')

    # RWS: midnight observations
    if _stn in wl_obs_dfs:
        _obs = wl_obs_dfs[_stn][wl_obs_dfs[_stn]['timestamp'].dt.year == 2015].copy()
        ax.scatter(_obs['timestamp'], _obs['wl_m'],
                   s=20, color='tab:orange', alpha=0.85, linewidths=0,
                   label='RWS obs (00:00, m NAP)', zorder=3)

    _units = ds_2015['elev'].attrs.get('units', 'm')
    ax.set_xlim(pd.Timestamp('2015-01-01'), pd.Timestamp('2016-01-01'))
    ax.set_ylabel(f'Water level [{_units}]')
    ax.set_title(f"{_stn} — midnight elevation  "
                 f"(lon={_meta['lon']:.3f}°, lat={_meta['lat']:.3f}°)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper right')
    ax.set_xlabel('Time (2015)')

fig.suptitle('Sea surface height at 00:00: EMOaaS vs TrilaWatt vs RWS (2015)',
             fontsize=13, fontweight='bold')
fig.autofmt_xdate()
plt.show()

In [ ]:
# Scatter: RWS obs (x) vs EMOaaS & TrilaWatt (y) at midnight — per station

fig, axes = plt.subplots(1, len(STATIONS_WL), figsize=(6 * len(STATIONS_WL), 5),
                         constrained_layout=True)
axes = np.atleast_1d(axes)

for _idx, (_stn, _meta) in enumerate(STATIONS_WL.items()):
    ax = axes[_idx]

    if _stn not in wl_obs_dfs:
        ax.set_title(f"{_stn} — no obs"); continue

    _obs = wl_obs_dfs[_stn][wl_obs_dfs[_stn]['timestamp'].dt.year == 2015].copy()
    if _obs.empty:
        ax.set_title(f"{_stn} — no 2015 obs"); continue

    # Match each RWS timestamp to nearest EMOaaS timestep
    _, _nn = _tree_wl.query([[_meta['lon'], _meta['lat']]])
    _fi = _vi_wl[_nn[0]]
    _iy, _ix = np.unravel_index(_fi, _lon2d_wl.shape)
    _model_ts    = _elev_da.isel({_hy_wl: int(_iy), _hx_wl: int(_ix)})
    _model_times = pd.DatetimeIndex(pd.to_datetime(_model_ts[_td_wl].values))
    _model_vals  = _model_ts.values
    _t_idx       = np.array([int(np.argmin(np.abs(_model_times - t))) for t in _obs['timestamp']])
    _emo_matched = _model_vals[_t_idx]

    ax.scatter(_obs['wl_m'].values, _emo_matched,
               s=20, color='tab:blue', alpha=0.7, linewidths=0, label='EMOaaS', zorder=3)

    _all_vals = np.concatenate([_obs['wl_m'].values, _emo_matched])

    # Match each RWS timestamp to nearest TrilaWatt timestep
    if 'sea_surface_height_2d' in ds_tw.variables:
        _ti = int(np.argmin(np.abs(ds_tw.lon.values - _meta['lon'])))
        _tj = int(np.argmin(np.abs(ds_tw.lat.values - _meta['lat'])))
        _tw_box = ds_tw['sea_surface_height_2d'].isel(
            lon=slice(max(0, _ti - 2), _ti + 3),
            lat=slice(max(0, _tj - 2), _tj + 3),
        ).mean(dim=['lon', 'lat'], skipna=True)
        _tw_df      = _tw_box.to_dataframe(name='ssh').dropna()
        _tw_2015    = _tw_df['ssh'][_tw_df.index.year == 2015]
        _tw_times   = pd.DatetimeIndex(_tw_2015.index)
        _tw_idx     = np.array([int(np.argmin(np.abs(_tw_times - t))) for t in _obs['timestamp']])
        _tw_matched = _tw_2015.values[_tw_idx]

        ax.scatter(_obs['wl_m'].values, _tw_matched,
                   s=20, color='tab:green', alpha=0.7, linewidths=0, label='TrilaWatt', zorder=2)
        _all_vals = np.concatenate([_all_vals, _tw_matched])

    # 1:1 reference line
    _finite = _all_vals[np.isfinite(_all_vals)]
    _lim = [_finite.min() - 0.05, _finite.max() + 0.05]
    ax.plot(_lim, _lim, 'k--', lw=1.0, label='1:1')
    ax.set_xlim(_lim); ax.set_ylim(_lim); ax.set_aspect('equal')

    _units = ds_2015['elev'].attrs.get('units', 'm')
    ax.set_xlabel(f'RWS obs [{_units}] (m NAP)')
    ax.set_ylabel(f'Model [{_units}]')
    ax.set_title(_stn)
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper left')

fig.suptitle('Water level scatter: model vs RWS obs at 00:00 (2015)',
             fontsize=13, fontweight='bold')
plt.show()


In [ ]:
# Salinity validation: RWS obs vs EMOaaS (salt) and TrilaWatt (sea_water_salinity_2d)
# RWS CSV: WAARNEMINGDATUM (DD-MM-YYYY), WAARNEMINGTIJD (HH:MM:SS),
#          NUMERIEKEWAARDE (psu), LAT, LON (decimal degrees)

# Model outputs filtered to 00:00:00 (midnight) 
# Field measurements keep time as-is (not filtered to midnight) to match the RWS dataset.

salinity_2015_dws        = 'salinity_2015_dws.csv'




from scipy.spatial import cKDTree

sal_fpath = Validation_DATA_DIR / 'Field' / 'salinity_2015_dws.csv'
if not sal_fpath.exists():
    raise FileNotFoundError(f"Salinity file not found: {sal_fpath}")

_df_sal = pd.read_csv(sal_fpath, na_values=['NA', ''], sep=';')
_df_sal['timestamp'] = pd.to_datetime(
    _df_sal['WAARNEMINGDATUM'].astype(str) + ' ' + _df_sal['WAARNEMINGTIJD'].astype(str),
    dayfirst=True, errors='coerce'
)
_df_sal = _df_sal.dropna(subset=['timestamp', 'NUMERIEKEWAARDE', 'LAT', 'LON'])
_df_sal['sal_psu'] = _df_sal['NUMERIEKEWAARDE']
_df_sal = _df_sal[_df_sal['timestamp'].dt.year == 2015].sort_values('timestamp').reset_index(drop=True)
print(f"RWS salinity: {len(_df_sal)} observations in 2015")

# Build KDTree on EMOaaS model grid
_ln_sal    = _pick_coord_name(ds_2015, ('lonc', 'lon', 'longitude'))
_la_sal    = _pick_coord_name(ds_2015, ('latc', 'lat', 'latitude'))
_lon2d_sal = ds_2015[_ln_sal].values
_lat2d_sal = ds_2015[_la_sal].values
_lf_sal    = _lon2d_sal.ravel()
_ltf_sal   = _lat2d_sal.ravel()
_vm_sal    = np.isfinite(_lf_sal) & np.isfinite(_ltf_sal)
_vi_sal    = np.where(_vm_sal)[0]
_tree_sal  = cKDTree(np.column_stack([_lf_sal[_vm_sal], _ltf_sal[_vm_sal]]))

# Prepare EMOaaS salt DataArray
if 'salt' not in ds_2015.variables:
    raise KeyError(f"'salt' not in model; available: {sorted(ds_2015.data_vars)}")
_salt_da    = ds_2015['salt'].squeeze(drop=True)
_td_sal     = _find_time_dim(_salt_da)
_zd_sal     = _find_vertical_dim(_salt_da, _td_sal)
if _zd_sal is not None:
    _salt_da = _salt_da.isel({_zd_sal: MODEL_SURFACE_LAYER_INDEX})
_salt_da    = _drop_duplicate_time(_salt_da, _td_sal)
_hy_sal, _hx_sal = [d for d in _salt_da.dims if d != _td_sal]
_salt_times = pd.DatetimeIndex(pd.to_datetime(_salt_da[_td_sal].values))

# Match each RWS obs to nearest EMOaaS grid cell + nearest timestep
_, _nn_emo = _tree_sal.query(_df_sal[['LON', 'LAT']].values)
_fi_emo    = _vi_sal[_nn_emo]
_iy_emo, _ix_emo = np.unravel_index(_fi_emo, _lon2d_sal.shape)
_t_emo     = np.array([int(np.argmin(np.abs(_salt_times - t))) for t in _df_sal['timestamp']])
_df_sal['emo_sal'] = [
    float(_salt_da.isel({_hy_sal: int(iy), _hx_sal: int(ix), _td_sal: int(it)}).values)
    for iy, ix, it in zip(_iy_emo, _ix_emo, _t_emo)
]
print(f"EMOaaS matching done")

# Match each RWS obs to nearest TrilaWatt grid cell + nearest timestep
if 'sea_water_salinity_2d' in ds_tw.variables:
    _tw_lon_1d = ds_tw.lon.values
    _tw_lat_1d = ds_tw.lat.values
    _tw_lon2d_g, _tw_lat2d_g = np.meshgrid(_tw_lon_1d, _tw_lat_1d)  # shape (n_lat, n_lon)
    _tw_lf_g   = _tw_lon2d_g.ravel()
    _tw_ltf_g  = _tw_lat2d_g.ravel()
    _tw_vm_g   = np.isfinite(_tw_lf_g) & np.isfinite(_tw_ltf_g)
    _tw_vi_g   = np.where(_tw_vm_g)[0]
    _tree_tw_sal = cKDTree(np.column_stack([_tw_lf_g[_tw_vm_g], _tw_ltf_g[_tw_vm_g]]))

    _tw_sal_da    = ds_tw['sea_water_salinity_2d']
    _tw_times_sal = pd.DatetimeIndex(pd.to_datetime(_tw_sal_da.time.values))

    _, _nn_tw = _tree_tw_sal.query(_df_sal[['LON', 'LAT']].values)
    _fi_tw    = _tw_vi_g[_nn_tw]
    _iy_tw, _ix_tw = np.unravel_index(_fi_tw, _tw_lon2d_g.shape)
    _t_tw     = np.array([int(np.argmin(np.abs(_tw_times_sal - t))) for t in _df_sal['timestamp']])
    _df_sal['tw_sal'] = [
        float(_tw_sal_da.isel(lat=int(iy), lon=int(ix), time=int(it)).values)
        for iy, ix, it in zip(_iy_tw, _ix_tw, _t_tw)
    ]
    print(f"TrilaWatt matching done")
else:
    _df_sal['tw_sal'] = np.nan

_df_sal = _df_sal.dropna(subset=['sal_psu', 'emo_sal'])
_sal_units = ds_2015['salt'].attrs.get('units', 'psu')

# ── Time series ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 9), constrained_layout=True)

ax = axes[0]
ax.scatter(_df_sal['timestamp'], _df_sal['sal_psu'],
           s=8, color='tab:orange', alpha=0.6, linewidths=0, label='RWS obs', zorder=3)
ax.scatter(_df_sal['timestamp'], _df_sal['emo_sal'],
           s=8, color='tab:blue', alpha=0.6, linewidths=0, label='EMOaaS (nearest point)', zorder=2)
if _df_sal['tw_sal'].notna().any():
    ax.scatter(_df_sal['timestamp'], _df_sal['tw_sal'],
               s=8, color='tab:green', alpha=0.5, linewidths=0, label='TrilaWatt (nearest point)', zorder=1)
ax.set_xlim(pd.Timestamp('2015-01-01'), pd.Timestamp('2016-01-01'))
ax.set_ylabel(f'Salinity [{_sal_units}]')
ax.set_title('Salinity time series — each obs matched to nearest model grid point')
ax.grid(True, alpha=0.25)
ax.legend(loc='upper right')
ax.set_xlabel('Time (2015)')

# ── Scatter ────────────────────────────────────────────────────────────────
ax2 = axes[1]
ax2.scatter(_df_sal['sal_psu'], _df_sal['emo_sal'],
            s=12, color='tab:blue', alpha=0.6, linewidths=0, label='EMOaaS', zorder=3)
if _df_sal['tw_sal'].notna().any():
    ax2.scatter(_df_sal['sal_psu'], _df_sal['tw_sal'],
                s=12, color='tab:green', alpha=0.6, linewidths=0, label='TrilaWatt', zorder=2)

_all_sal = np.concatenate([_df_sal['sal_psu'].values, _df_sal['emo_sal'].values,
                            _df_sal['tw_sal'].dropna().values])
_all_sal  = _all_sal[np.isfinite(_all_sal)]
_sal_lim  = [_all_sal.min() - 0.5, _all_sal.max() + 0.5]
ax2.plot(_sal_lim, _sal_lim, 'k--', lw=1.0, label='1:1')
ax2.set_xlim(_sal_lim); ax2.set_ylim(_sal_lim); ax2.set_aspect('equal')
ax2.set_xlabel(f'RWS obs [{_sal_units}]')
ax2.set_ylabel(f'Model [{_sal_units}]')
ax2.set_title('Salinity scatter: model vs RWS obs (2015)')
ax2.grid(True, alpha=0.25)
ax2.legend(loc='upper left')

fig.suptitle('Salinity validation: EMOaaS vs TrilaWatt vs RWS (2015)',
             fontsize=13, fontweight='bold')
fig.autofmt_xdate()
plt.show()

print(f"Total matched observations: {len(_df_sal)}")



In [ ]:
DATA_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/porosity_universal/')
spinup_datasets = {}

for i in [1,2]:
    Spinup_DIR = DATA_DIR / f"spinup_{i:02d}"
    files = sorted(Spinup_DIR.glob(FILE_PATTERN))

    if not files:
        raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {Spinup_DIR}')

    ds = xr.open_mfdataset(
        files,
        combine='nested',
        concat_dim='time',
        decode_times=True,
        data_vars='minimal',
        coords='minimal',
        compat='override',
        join='override',
    )

    spinup_datasets[f"spinup_{i:02d}"] = ds


In [ ]:
# Goal: show the selected subdomain on a bathymetry map.
bathy_name = _find_bathy_name(ds, Bathymetry_VAR)
bathy = ds[bathy_name].squeeze(drop=True)

# Find horizontal dims from bathymetry; if a time dim exists, use first timestep.
bathy_dims = list(bathy.dims)
time_like = [d for d in bathy_dims if 'time' in d.lower()]
if time_like:
    bathy = bathy.isel({time_like[0]: 0})
    bathy_dims = [d for d in bathy.dims if d != time_like[0]]

if len(bathy_dims) != 2:
    raise ValueError(f'Expected 2D bathymetry after squeezing, got dims {bathy.dims}')
y_dim, x_dim = bathy_dims

ny = bathy.sizes[y_dim]
nx = bathy.sizes[x_dim]
y0, y1 = Y_SLICE
x0, x1 = X_SLICE
if not (0 <= y0 < y1 <= ny):
    raise IndexError(f'Y_SLICE={Y_SLICE} out of bounds for {y_dim} size {ny}')
if not (0 <= x0 < x1 <= nx):
    raise IndexError(f'X_SLICE={X_SLICE} out of bounds for {x_dim} size {nx}')

lon_name = _pick_coord_name(ds, ('lonc', 'lon', 'longitude'))
lat_name = _pick_coord_name(ds, ('latc', 'lat', 'latitude'))

use_geo = False
if lon_name is not None and lat_name is not None:
    lon_da = ds[lon_name]
    lat_da = ds[lat_name]
    if y_dim in lon_da.dims and x_dim in lon_da.dims and y_dim in lat_da.dims and x_dim in lat_da.dims:
        x_plot = _to_float(lon_da.transpose(y_dim, x_dim).values)
        y_plot = _to_float(lat_da.transpose(y_dim, x_dim).values)
        if x_plot.shape == (ny, nx) and y_plot.shape == (ny, nx):
            if np.isfinite(x_plot).all() and np.isfinite(y_plot).all():
                use_geo = True

if not use_geo:
    x_plot, y_plot = np.meshgrid(np.arange(nx, dtype=float), np.arange(ny, dtype=float))

bathy2d = _to_float(bathy.transpose(y_dim, x_dim).values)

fig, ax = plt.subplots(figsize=(8.6, 6.8))
mesh = ax.pcolormesh(
    x_plot,
    y_plot,
    np.ma.masked_invalid(bathy2d),
    shading='auto',
    cmap='cividis',
)
cbar = fig.colorbar(mesh, ax=ax, fraction=0.04, pad=0.03)
units = bathy.attrs.get('units', '')
cbar.set_label(f'{bathy_name} [{units}]' if units else bathy_name)

# Draw the selected subdomain as a closed polygon on the map.
if use_geo:
    x_poly = [x_plot[y0, x0], x_plot[y0, x1 - 1], x_plot[y1 - 1, x1 - 1], x_plot[y1 - 1, x0], x_plot[y0, x0]]
    y_poly = [y_plot[y0, x0], y_plot[y0, x1 - 1], y_plot[y1 - 1, x1 - 1], y_plot[y1 - 1, x0], y_plot[y0, x0]]
    ax.plot(x_poly, y_poly, color='red', lw=2.2, label='Selected subdomain')
else:
    x_poly = [x0, x1, x1, x0, x0]
    y_poly = [y0, y0, y1, y1, y0]
    ax.plot(x_poly, y_poly, color='red', lw=2.2, label='Selected subdomain')

ax.set_title('Subdomain location on bathymetry map')
ax.set_xlabel('Longitude' if use_geo else x_dim)
ax.set_ylabel('Latitude' if use_geo else y_dim)
ax.grid(True, alpha=0.2)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f'Bathymetry variable used: {bathy_name}')
print(f'Subdomain used: y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}')

In [ ]:
fig, axes = plt.subplots(
    len(vars_list),
    1,
    figsize=(14, 3 * len(vars_list)),
    sharex=True,
    constrained_layout=True,
)

if len(vars_list) == 1:
    axes = [axes]

failed_vars = []
missing_vars = []

for ax, vname in zip(axes, vars_list):

    all_values = []
    boundaries = [0]
    spin_labels = []
    units = ""

    for spin_name in sorted(spinup_datasets.keys()):

        ds = spinup_datasets[spin_name]

        try:

            if vname not in ds.variables:
                missing_vars.append((spin_name, vname))
                continue

            da = ds[vname].squeeze(drop=True)

            time_dim = _find_time_dim(da)
            z_dim = _find_vertical_dim(da, time_dim)

            # Select surface layer if present
            if z_dim is not None:
                da = da.isel({z_dim: SURFACE_LAYER_INDEX})

            # Remove duplicate timestamps
            da = _drop_duplicate_time(da, time_dim)

            # Spatial average
            spatial_dims = [d for d in da.dims if d != time_dim]

            if len(spatial_dims) == 0:
                raise ValueError("No spatial dimensions available.")

            # Variable-specific masking
            if vname.lower() in ("chla", "netppm2"):
                da = da.where(da >= 0)

            if vname.lower() == "etw":
                da = da.where(da >= -10)

            if vname.lower() == "xEPS":
                da = da.where(da >= 0)

            series = da.mean(dim=spatial_dims, skipna=True)

            # Daily mean / rolling mean
            series = _maybe_smooth(series, time_dim)

            if units == "":
                units = da.attrs.get("units", "")

            values = series.values

            all_values.append(values)

            boundaries.append(boundaries[-1] + len(values))
            spin_labels.append(spin_name)

        except Exception as e:
            failed_vars.append((spin_name, vname, str(e)))

    # ----------------------------------------------------
    # Concatenate all spinups into one long time series
    # ----------------------------------------------------
    long_series = np.concatenate(all_values)
    x = np.arange(len(long_series))

    ax.plot(x, long_series, lw=1.4, color="tab:blue")

    # Draw spinup boundaries
    ymax = np.nanmax(long_series)
    ymin = np.nanmin(long_series)

    for i, b in enumerate(boundaries[:-1]):

        ax.axvline(b, color="gray", ls="--", lw=0.8, alpha=0.6)

        # Put spinup label in the middle of each segment
        if i < len(spin_labels):
            center = (boundaries[i] + boundaries[i+1]) / 2
            ax.text(
                center,
                ymax,
                spin_labels[i],
                ha="center",
                va="bottom",
                fontsize=8,
            )

    ylabel = f"{vname} [{units}]" if units else vname

    ax.set_ylabel(ylabel)
    ax.set_title(vname)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Model timestep (concatenated spinups)")

plt.show()

In [ ]:
csv_path = Validation_DATA_DIR / Marsdiep_ts

if not csv_path.exists():
    raise FileNotFoundError(f'CSV file not found: {csv_path}')

measurement_df = pd.read_csv(
    csv_path,
    na_values=['NA', ''],
    parse_dates=['timestamp'],
)

print(f'Loaded CSV: {csv_path}')
print(f'Shape: {measurement_df.shape[0]} rows x {measurement_df.shape[1]} columns')
print('Columns:')
print(list(measurement_df.columns))
print('')
print('First 5 rows:')
display(measurement_df.head())



In [ ]:
# Goal: Compare model results with observations for temporal dynamics of Chla and other variables nearby Marsdiep.


#Obs_VAR_NAME = 'TSM'
#Model_VAR_NAME = 'ESS'

#Obs_VAR_NAME = 'Daily_PP'
#Model_VAR_NAME = 'netPPm2'

#Obs_VAR_NAME = 'POC'
#Model_VAR_NAME = 'R6c'

Obs_VAR_NAME = 'NH4'
Model_VAR_NAME = 'N4n'

if Model_VAR_NAME not in ds.variables:
    raise KeyError(f"Variable '{Model_VAR_NAME}' not found. Available: {sorted(ds.data_vars)}")

Model_ds = ds[Model_VAR_NAME].squeeze(drop=True)
time_dim = _find_time_dim(Model_ds)

z_dim = _find_vertical_dim(Model_ds, time_dim)   # <-- may be None for 2D variables

# ---------------------------------------------------------
# 1. Handle 2D vs 3D variables
# ---------------------------------------------------------
if z_dim is None:
    # 2D variable: dims = (time, y, x)
    spatial_dims = [d for d in Model_ds.dims if d != time_dim]
    if len(spatial_dims) != 2:
        raise ValueError(f"Expected 2D variable with dims (time,y,x), got {Model_ds.dims}")
    y_dim, x_dim = spatial_dims
    Model_sub = Model_ds.isel({y_dim: slice(Y_SLICE[0], Y_SLICE[1]),
                               x_dim: slice(X_SLICE[0], X_SLICE[1])})
else:
    # 3D variable: dims = (time, z, y, x)
    xy_dims = [d for d in Model_ds.dims if d not in (time_dim, z_dim)]
    if len(xy_dims) != 2:
        raise ValueError(f'Expected 2 horizontal dims for {Model_VAR_NAME}, got {xy_dims}')
    y_dim, x_dim = xy_dims

    Model_sub = Model_ds.isel({
        z_dim: MODEL_SURFACE_LAYER_INDEX,
        y_dim: slice(Y_SLICE[0], Y_SLICE[1]),
        x_dim: slice(X_SLICE[0], X_SLICE[1])
    })

# ---------------------------------------------------------
# 2. Compute spatial mean
# ---------------------------------------------------------
model_series = Model_sub.mean(dim=(y_dim, x_dim), skipna=True)
model_series = _drop_duplicate_time(model_series, time_dim)
model_series = _maybe_smooth(model_series, time_dim)
model_series = model_series.where(model_series >= 0)

# Day of year
model_series['doy'] = model_series[time_dim].dt.dayofyear

# ---------------------------------------------------------
# 3. Observations
# ---------------------------------------------------------
if Obs_VAR_NAME not in measurement_df.columns:
    raise KeyError(f"Column {Obs_VAR_NAME} not found in the file.")

obs_df = measurement_df[['timestamp', Obs_VAR_NAME]].dropna(subset=['timestamp', Obs_VAR_NAME]).copy()
obs_df['timestamp'] = pd.to_datetime(obs_df['timestamp'], dayfirst=True, errors='coerce')
obs_df = obs_df.sort_values('timestamp')

if obs_df.empty:
    raise ValueError('No measurements found.')

obs_df['doy'] = obs_df['timestamp'].dt.dayofyear

# ---------------------------------------------------------
# 4. Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 4.8))

ax.scatter(
    model_series['doy'],
    model_series.values,
    s=18,
    color='tab:blue',
    label=f'Model {Model_VAR_NAME}',
)

ax.scatter(
    obs_df['doy'],
    #obs_df['POC'] * 1000, # for POC to be mg/m3
    obs_df[Obs_VAR_NAME],
    s=22,
    color='tab:orange',
    alpha=0.85,
    label=f'{Obs_VAR_NAME}',
    zorder=3,
)

Var_units = Model_ds.attrs.get('units', '')
ax.set_ylabel(f'{Obs_VAR_NAME} [{Var_units}]')
ax.set_xlabel('Day of Year')
ax.set_title(
    f'Model vs observed {Obs_VAR_NAME} | '
    f'y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}'
)
ax.grid(True, alpha=0.25)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(f'Subdomain used: y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}')
print(f'Number of observation points: {len(obs_df)}')


In [ ]:
# Compare model surface-layer subdomain means with observed surface measurements for:
# Chla (Chl), Phosphate (N1p vs PO4), Nitrate (N3n vs NO3), Ammonium (N4n vs NH4)

import pandas as pd
import matplotlib.pyplot as plt

MODEL_SURFACE_LAYER_INDEX = 10  # top=11, bottom=1 in your convention
USE_DAILY_MEAN = False
ROLLING_WINDOW = None  # e.g. 3 for smoothing, or None

# Model variable -> observation column
VAR_MAP = {
    "Chla": "Chl",
    #"R6c": "POC",
    "ESS": "TSM",
    #"N1p": "PO4",
    "N3n": "NO3",
    "N4n": "NH4",
    "netPPm2": "",
}

# Reuse opened dataset if available; otherwise open yearly files.
try:
    ds
except NameError:
    files = sorted(DATA_DIR.glob(FILE_PATTERN))
    if not files:
        raise FileNotFoundError(f"No files found with pattern: {FILE_PATTERN} in {DATA_DIR}")
    ds = xr.open_mfdataset(
        files,
        combine="nested",
        concat_dim="time",
        decode_times=True,
        data_vars="minimal",
        coords="minimal",
        compat="override",
        join="override",
    )

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if "time" in d.lower():
            return d
    raise ValueError(f"No time dimension found in {da.dims}")

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ("level", "z", "sigma", "layer", "lev", "depth", "nmesh2_layer_3d")
    for d in da.dims:
        if d != time_dim and any(k in d.lower() for k in candidates):
            return d
    return None

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

def _maybe_smooth(series: xr.DataArray, time_dim: str) -> xr.DataArray:
    out = series
    if USE_DAILY_MEAN:
        out = out.resample({time_dim: "1D"}).mean(skipna=True)
    if ROLLING_WINDOW is not None:
        if ROLLING_WINDOW < 1:
            raise ValueError("ROLLING_WINDOW must be >= 1 or None")
        out = out.rolling({time_dim: ROLLING_WINDOW}, center=True).mean()
    return out

def _model_surface_series(ds: xr.Dataset, model_var: str, layer_index: int) -> tuple[xr.DataArray, str]:
    if model_var not in ds.variables:
        raise KeyError(f"Model variable '{model_var}' not found. Available: {sorted(ds.data_vars)}")

    da = ds[model_var].squeeze(drop=True)
    time_dim = _find_time_dim(da)
    z_dim = _find_vertical_dim(da, time_dim)
    if z_dim is None:
        raise ValueError(f"No vertical dimension found for {model_var}. Dims: {da.dims}")

    xy_dims = [d for d in da.dims if d not in (time_dim, z_dim)]
    if len(xy_dims) != 2:
        raise ValueError(f"Expected 2 horizontal dims for {model_var}, got {xy_dims} from {da.dims}")
    y_dim, x_dim = xy_dims

    y0, y1 = Y_SLICE
    x0, x1 = X_SLICE
    if not (0 <= y0 < y1 <= da.sizes[y_dim]):
        raise IndexError(f"Y_SLICE={Y_SLICE} out of bounds for {y_dim} size {da.sizes[y_dim]}")
    if not (0 <= x0 < x1 <= da.sizes[x_dim]):
        raise IndexError(f"X_SLICE={X_SLICE} out of bounds for {x_dim} size {da.sizes[x_dim]}")

    if layer_index >= da.sizes[z_dim] or layer_index < -da.sizes[z_dim]:
        raise IndexError(f"layer_index={layer_index} out of bounds for '{z_dim}' size {da.sizes[z_dim]}")

    sub = da.isel({y_dim: slice(y0, y1), x_dim: slice(x0, x1)})
    series = sub.isel({z_dim: layer_index}).mean(dim=(y_dim, x_dim), skipna=True)
    series = _drop_duplicate_time(series, time_dim)
    series = _maybe_smooth(series, time_dim)
    return series, time_dim

# Observations: reuse measurement_df if present, otherwise load it.
try:
    measurement_df
except NameError:
    Validation_DATA_DIR = Path("/export/lv9/user/qzhan/archived_output/validation_data/")
    Marsdiep_ts = "pelagic/ts/Jetty_HWseries.csv"
    csv_path = Validation_DATA_DIR / Marsdiep_ts
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")
    measurement_df = pd.read_csv(
        csv_path,
        na_values=["NA", ""],
        parse_dates=["timestamp"],
    )

if "timestamp" not in measurement_df.columns:
    raise KeyError("Column 'timestamp' not found in observations CSV.")

obs_df = measurement_df.copy()
obs_df = obs_df.dropna(subset=["timestamp"])
obs_df = obs_df[obs_df["timestamp"].dt.year == 2015].sort_values("timestamp")

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
axes = axes.ravel()

for i, (model_var, obs_col) in enumerate(VAR_MAP.items()):
    ax = axes[i]

    if obs_col not in obs_df.columns:
        ax.text(0.5, 0.5, f"Obs column '{obs_col}' not found", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{model_var} vs {obs_col}")
        ax.set_axis_off()
        continue

    model_series, time_dim = _model_surface_series(ds, model_var, MODEL_SURFACE_LAYER_INDEX)
    model_series = model_series.where(model_series >= 0 if model_var == "Chla" else True)

    obs_var = obs_df[["timestamp", obs_col]].dropna()

    ax.plot(
        model_series[time_dim].values,
        model_series.values,
        lw=2.0,
        color="tab:blue",
        label=f"Model {model_var} (layer {MODEL_SURFACE_LAYER_INDEX})",
    )
    ax.scatter(
        obs_var["timestamp"],
        obs_var[obs_col],
        s=18,
        color="tab:orange",
        alpha=0.85,
        label=f"Obs {obs_col}",
        zorder=3,
    )

    units = ds[model_var].attrs.get("units", "")
    ax.set_ylabel(f"{model_var} / {obs_col} [{units}]" if units else f"{model_var} / {obs_col}")
    ax.set_title(f"{model_var} (model) vs {obs_col} (obs)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper left", fontsize=8)

for ax in axes:
    ax.set_xlabel("Time")

fig.suptitle(
    f"Model vs observed surface concentrations in 2015 | y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}",
    y=1.02,
)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Compare model subdomain means with observed measurements for:
# Primary production
if not PP_csv_path.exists():
    raise FileNotFoundError(f'CSV file not found: {PP_csv_path}')

pp_df = pd.read_csv(PP_csv_path, na_values=['NA', ''])
if 'Date' not in pp_df.columns or 'Daily_PP' not in pp_df.columns:
    raise KeyError("CSV must contain columns 'Date' and 'Daily_PP'.")

pp_df['Date'] = pd.to_datetime(pp_df['Date'], errors='coerce')
pp_obs = pp_df[['Date', 'Daily_PP']].dropna(subset=['Date', 'Daily_PP']).copy()
pp_obs = pp_obs.sort_values('Date')
pp_obs = pp_obs[pp_obs['Date'].dt.year == 2015]

if pp_obs.empty:
    raise ValueError('No Daily_PP observations found for 2015 in PP CSV.')

# Reuse opened dataset if available; otherwise open yearly files.
try:
    ds
except NameError:
    files = sorted(DATA_DIR.glob(FILE_PATTERN))
    if not files:
        raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {DATA_DIR}')
    ds = xr.open_mfdataset(
        files,
        combine='nested',
        concat_dim='time',
        decode_times=True,
        data_vars='minimal',
        coords='minimal',
        compat='override',
        join='override',
    )

MODEL_PP_VAR = 'netPPm2'
if MODEL_PP_VAR not in ds.variables:
    raise KeyError(f"Variable '{MODEL_PP_VAR}' not found. Available: {sorted(ds.data_vars)}")

pp_model_da = ds[MODEL_PP_VAR].squeeze(drop=True)

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time dimension found in {da.dims}')

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

time_dim = _find_time_dim(pp_model_da)
spatial_dims = [d for d in pp_model_da.dims if d != time_dim]

if len(spatial_dims) >= 2:
    # Use same subdomain as previous cells when 2D horizontal dims exist.
    y_dim, x_dim = spatial_dims[-2], spatial_dims[-1]
    y0, y1 = Y_SLICE
    x0, x1 = X_SLICE
    if 0 <= y0 < y1 <= pp_model_da.sizes[y_dim] and 0 <= x0 < x1 <= pp_model_da.sizes[x_dim]:
        pp_model_da = pp_model_da.isel({y_dim: slice(y0, y1), x_dim: slice(x0, x1)})

spatial_dims = [d for d in pp_model_da.dims if d != time_dim]
if spatial_dims:
    pp_model_series = pp_model_da.mean(dim=spatial_dims, skipna=True)
else:
    pp_model_series = pp_model_da

pp_model_series = _drop_duplicate_time(pp_model_series, time_dim)
# Remove physically invalid negative model PP values
pp_model_series = pp_model_series.where(pp_model_series >= 0)

# Match observation cadence: daily mean from model.
pp_model_daily = pp_model_series.resample({time_dim: '1D'}).mean(skipna=True)
pp_model_df = pd.DataFrame({
    'Date': pd.to_datetime(pp_model_daily[time_dim].values),
    'Model_PP': pp_model_daily.values
})
pp_model_df = pp_model_df.dropna(subset=['Date', 'Model_PP'])
pp_model_df = pp_model_df[pp_model_df['Date'].dt.year == 2015]

if pp_model_df.empty:
    raise ValueError('No model netPPm2 values found for 2015.')

# Plot model and observations.
fig, ax = plt.subplots(figsize=(12, 4.8))

ax.plot(
    pp_model_df['Date'],
    pp_model_df['Model_PP'],
    lw=2.0,
    color='tab:blue',
    label='Model netPPm2 (daily mean)',
)
ax.scatter(
    pp_obs['Date'],
    pp_obs['Daily_PP'],
    s=24,
    color='tab:orange',
    alpha=0.9,
    label='Observed Daily_PP',
    zorder=3,
)

pp_units = ds[MODEL_PP_VAR].attrs.get('units', '')
ax.set_ylabel(f'PP [{pp_units}]' if pp_units else 'PP')
ax.set_xlabel('Time')
ax.set_title(
    f'Model vs observed primary production in 2015 | y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}'
)
ax.grid(True, alpha=0.25)
ax.legend(loc='upper left')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f'PP CSV used: {PP_csv_path}')
print(f'Observation points in 2015: {len(pp_obs)}')
print(f'Model daily points in 2015: {len(pp_model_df)}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cmocean
import cartopy.crs as ccrs
import contextily as ctx
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Path to your Excel file
excel_path = "/export/lv9/projects/dws/results/validation/pelagic/SecchiDepth_trawllist241120_IngridTulp.xlsx"

# Load only specific sheet
df = pd.read_excel(excel_path, sheet_name="trawllist_241120")
df["Kd"] = 1.476/df["WATER_VISIBILITY"]+0.3541 # Jacobs et al. (2020) # Obs formula: Kd = 1.476 / Z_Secchi + 0.3541  (Jacobs et al. 2020)
print(df.head())



In [ ]:
# ── Settings ──────────────────────────────────────────────────────────────
MODEL_VAR = "temp"       # light extinction coefficient in the model (m⁻¹)
OBS_VAR   = "temperature"  # observed water temperature column in df_2015
SPINUP_KEY   = "spinup_01"  # which spinup run to use; adjust as needed
Unit = "°C"

v_min = -1 # -100%
v_max = 1 # 100%

In [ ]:
# ── Settings ──────────────────────────────────────────────────────────────
MODEL_VAR = "xEPS"       # light extinction coefficient in the model (m⁻¹)
OBS_VAR   = "Kd"  # observed water temperature column in df_2015
SPINUP_KEY   = "spinup_10"  # which spinup run to use; adjust as needed
Unit = "m⁻¹"

v_min = -3
v_max = 3

In [ ]:
# Compare modelled Temperature (temp) with observed temperature (2015)

import numpy as np
import matplotlib.pyplot as plt
import cmocean
from scipy.spatial import cKDTree

# ── Load model Kd ─────────────────────────────────────────────────────────
ds_model = spinup_datasets.get(SPINUP_KEY, ds)

if MODEL_VAR not in ds_model.variables:
    raise KeyError(
        f"Variable '{MODEL_VAR}' not found in '{SPINUP_KEY}'. "
        f"Available: {sorted(ds_model.data_vars)}"
    )

model_da = ds_model[MODEL_VAR].squeeze(drop=True)
td    = _find_time_dim(model_da)
zd    = _find_vertical_dim(model_da, td)

# Select surface layer if variable is 3D
if zd is not None:
    model_da = model_da.isel({zd: SURFACE_LAYER_INDEX})

model_da = _drop_duplicate_time(model_da, td)

# Infer horizontal dims from model_da (may differ from the global y_dim / x_dim)
h_dims = [d for d in model_da.dims if d != td]
if len(h_dims) != 2:
    raise ValueError(f"Expected 2 horizontal dims after selecting surface, got: {model_da.dims}")
kd_y_dim, kd_x_dim = h_dims

# ── KD-tree: build only from valid (non-NaN) grid points ─────────────────
lon2d = lon_da.values  # (ny, nx)
lat2d = lat_da.values

lon_flat  = lon2d.ravel()
lat_flat  = lat2d.ravel()
valid_mask   = np.isfinite(lon_flat) & np.isfinite(lat_flat)
valid_flat_idx = np.where(valid_mask)[0]   # indices into the full flat array

if valid_flat_idx.size == 0:
    raise ValueError("No valid (non-NaN) coordinates found in lon_da / lat_da.")

tree = cKDTree(np.column_stack([lon_flat[valid_mask], lat_flat[valid_mask]]))

# ── Observations: use df_2015 which already has the Kd column ─────────────
obs = df_2015[["date", "latitude_s", "longitude_s", OBS_VAR]].dropna().copy()
obs["date"] = pd.to_datetime(obs["date"])

model_times = pd.DatetimeIndex(model_da[td].values)

# ── Match each obs to the nearest valid model grid cell + nearest timestep ─
_, idx_in_valid = tree.query(obs[["longitude_s", "latitude_s"]].values)
flat_idx = valid_flat_idx[idx_in_valid]        # map back to full flat grid
iy_arr, ix_arr = np.unravel_index(flat_idx, lon2d.shape)

t_arr = np.array([int(np.argmin(np.abs(model_times - d))) for d in obs["date"]])

obs["var_model"] = [
    float(model_da.isel({kd_y_dim: int(iy), kd_x_dim: int(ix), td: int(it)}).values)
    for iy, ix, it in zip(iy_arr, ix_arr, t_arr)
]
# Remove rows where model Kd is negative (physically invalid) or NaN
obs["var_model"] = obs["var_model"].where(obs["var_model"] > 0)
obs = obs.dropna(subset=[OBS_VAR, "var_model"])
obs["diff"] = (obs["var_model"] - obs[OBS_VAR]) / obs[OBS_VAR]   # positive = model overestimates

print(f"Valid grid points used for KD-tree: {valid_flat_idx.size} / {lon_flat.size}")
print(f"Matched observation points: {len(obs)}")

# ── Scatter plot: obs vs model, coloured by (model − obs) ────────────
vabs = float(np.nanpercentile(np.abs(obs["diff"]), 95))   # robust symmetric limit


fig, ax = plt.subplots(figsize=(7, 6))

sc = ax.scatter(
    obs[OBS_VAR], obs["var_model"],
    c=obs["diff"],
    cmap=cmocean.cm.diff,
    vmin=v_min, vmax=v_max,
    s=45, edgecolors="k", linewidths=0.3, alpha=0.85, zorder=3,
)
cbar = plt.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label(f"Model {OBS_VAR.capitalize()} − Obs {OBS_VAR.capitalize()}  ({Unit})", fontsize=11)

# 1:1 reference line
all_vals = np.concatenate([obs[OBS_VAR].values, obs["var_model"].values])
pad = (all_vals.max() - all_vals.min()) * 0.05
lims = [all_vals.min() - pad, all_vals.max() + pad]
ax.plot(lims, lims, "k--", lw=1.2, label="1:1")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal")

ax.set_xlabel(f"Observed {OBS_VAR.capitalize()}  ({Unit})", fontsize=11)
ax.set_ylabel(f"Modelled {OBS_VAR.capitalize()}  ({Unit})", fontsize=11)
ax.set_title(f"Modelled vs Observed {OBS_VAR.capitalize()} – 2015  ({SPINUP_KEY})", fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")

# Summary statistics text box
n    = len(obs)
bias = obs["diff"].mean()
rmse = np.sqrt((obs["diff"] ** 2).mean())
corr = obs[[OBS_VAR, "var_model"]].corr().iloc[0, 1]

ax.text(
    0.05, 0.95,
    f"n = {n}\nBias = {bias:+.3f} \nRMSE = {rmse:.3f} \nR = {corr:.3f}",
    transform=ax.transAxes, va="top", fontsize=10,
    bbox=dict(facecolor="white", alpha=0.75, edgecolor="gray", boxstyle="round,pad=0.3"),
)

plt.tight_layout()
plt.show()


In [ ]:
# Water temperature validation: model vs in-situ measurements (2015)

# water temperature
#VMIN = 10.0
#VMAX = 25.0

# Kd
VMIN = 0
VMAX = 7
# ── Animation: all 2015 measurements ─────────────────────────────────────
df_2015 = (
    df[df["year"] == 2015]
    .dropna(subset=[OBS_VAR, "latitude_s", "longitude_s"])
    .copy()
)
df_2015["date"] = pd.to_datetime(df_2015[["year", "month", "day"]])
dates = sorted(df_2015["date"].unique())
print(f"\nUnique dates with valid water visibility in 2015: {len(dates)}")

fig_a, ax_a = plt.subplots(figsize=(9, 7), subplot_kw={"projection": PROJ})
ax_a.set_extent(MAP_EXTENT, crs=PROJ)
ctx.add_basemap(ax_a, crs=CRS_STR, source=BASEMAP_SRC, zorder=1)
gl_a = ax_a.gridlines(draw_labels=True, linewidth=0.4, color="gray", alpha=0.5)
gl_a.top_labels = False
gl_a.right_labels = False

# Anchor colorbar with a dummy scatter (keeps it fixed across frames)
dummy = ax_a.scatter(
    [], [], c=[], cmap=cmocean.cm.thermal, vmin=VMIN, vmax=VMAX,
    s=50, edgecolors="k", linewidths=0.3,
    transform=ccrs.PlateCarree(), zorder=3,
)
cbar_a = plt.colorbar(dummy, ax=ax_a, pad=0.02, shrink=0.7)
cbar_a.set_label(f"{OBS_VAR.capitalize()} ({Unit})", fontsize=11)
title_a = ax_a.set_title("", fontsize=13)

_scat = [None]  # mutable reference so update() can replace the artist

def _update(frame):
    date = dates[frame]
    df_f = df_2015[df_2015["date"] == date]
    if _scat[0] is not None:
        _scat[0].remove()
    _scat[0] = ax_a.scatter(
        df_f["longitude_s"], df_f["latitude_s"],
        c=df_f[OBS_VAR],
        cmap=cmocean.cm.thermal, vmin=VMIN, vmax=VMAX,
        s=50, edgecolors="k", linewidths=0.3,
        transform=ccrs.PlateCarree(), zorder=3,
    )
    title_a.set_text(
        f"{OBS_VAR.capitalize()} – {date.strftime('%Y-%m-%d')}  [{frame + 1}/{len(dates)}] – 2015  ({SPINUP_KEY})"
    )
    return _scat[0], title_a

anim = FuncAnimation(fig_a, _update, frames=len(dates), interval=500, blit=False)
plt.tight_layout()
HTML(anim.to_jshtml())

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import contextily as ctx
import cmocean

# ── Settings ──────────────────────────────────────────────────────────────
PROJ       = ccrs.PlateCarree()
CRS_STR    = PROJ.to_string()
BASEMAP    = ctx.providers.Esri.WorldShadedRelief
MAP_EXTENT = [3.5, 8.5, 52.5, 55.5]   # NL coastal region

# Use all 2015 matched points
df_all = obs.copy()

# Symmetric colour range for whole-year differences
vabs = float(np.nanpercentile(np.abs(df_all["diff"]), 95))

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": PROJ})
ax.set_extent(MAP_EXTENT, crs=PROJ)

# Basemap
ctx.add_basemap(ax, crs=CRS_STR, source=BASEMAP, zorder=1)

# Gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="gray", alpha=0.5)
gl.top_labels = False
gl.right_labels = False

# Scatter all points from the entire year
sc = ax.scatter(
    df_all["longitude_s"], df_all["latitude_s"],
    c=df_all["diff"],
    cmap=cmocean.cm.diff,
    vmin=v_min, vmax=v_max,
    s=55, edgecolors="k", linewidths=0.3,
    transform=ccrs.PlateCarree(), zorder=3,
)

cbar = plt.colorbar(sc, ax=ax, pad=0.02, shrink=0.7)
cbar.set_label(f"Model − Obs {OBS_VAR.capitalize()}  ({Unit})", fontsize=11)

ax.set_title(f"Model–Obs {OBS_VAR}  Difference (All 2015 Observations) ({SPINUP_KEY})", fontsize=14)
plt.tight_layout()
plt.show()
